# UK used car listings – cleaning & pricing model

Working through the staging table. Check what's missing, drop the exact
duplicate rows, sanity check the numeric ranges, build a clean analysis
table and then fit a few pricing models on top.

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

cars = datasets["uk_used_car_listings_staging"]
print("rows:", cars.shape[0])
print("cols:", cars.shape[1])
cars.head()

In [ ]:
# easier to profile this in SQL, so drop it into an in-memory sqlite db
conn = sqlite3.connect(":memory:")
cars.to_sql("used_cars", conn, index=False, if_exists="replace")

pd.read_sql_query("SELECT COUNT(*) AS total_rows FROM used_cars", conn)

## 1. Data profiling

In [ ]:
# nulls / blanks per column
missing_values = pd.read_sql_query("""
    SELECT 'manufacturer' AS field_name, SUM(CASE WHEN manufacturer IS NULL OR TRIM(manufacturer)='' THEN 1 ELSE 0 END) AS missing_count FROM used_cars
    UNION ALL SELECT 'model', SUM(CASE WHEN model IS NULL OR TRIM(model)='' THEN 1 ELSE 0 END) FROM used_cars
    UNION ALL SELECT 'year', SUM(CASE WHEN year IS NULL THEN 1 ELSE 0 END) FROM used_cars
    UNION ALL SELECT 'price', SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) FROM used_cars
    UNION ALL SELECT 'transmission', SUM(CASE WHEN transmission IS NULL OR TRIM(transmission)='' THEN 1 ELSE 0 END) FROM used_cars
    UNION ALL SELECT 'mileage', SUM(CASE WHEN mileage IS NULL THEN 1 ELSE 0 END) FROM used_cars
    UNION ALL SELECT 'fuel_type', SUM(CASE WHEN fuel_type IS NULL OR TRIM(fuel_type)='' THEN 1 ELSE 0 END) FROM used_cars
    UNION ALL SELECT 'tax', SUM(CASE WHEN tax IS NULL THEN 1 ELSE 0 END) FROM used_cars
    UNION ALL SELECT 'mpg', SUM(CASE WHEN mpg IS NULL THEN 1 ELSE 0 END) FROM used_cars
    UNION ALL SELECT 'engine_size', SUM(CASE WHEN engine_size IS NULL THEN 1 ELSE 0 END) FROM used_cars
    UNION ALL SELECT 'source_file', SUM(CASE WHEN source_file IS NULL OR TRIM(source_file)='' THEN 1 ELSE 0 END) FROM used_cars
""", conn)
missing_values["missing_percentage"] = (missing_values["missing_count"] / len(cars) * 100).round(2)
missing_values

In [ ]:
# how many exact-duplicate rows are there (everything except source_file)
duplicate_summary = pd.read_sql_query("""
    WITH grp AS (
        SELECT manufacturer, model, year, price, transmission, mileage,
               fuel_type, tax, mpg, engine_size, COUNT(*) AS record_count
        FROM used_cars
        GROUP BY manufacturer, model, year, price, transmission, mileage,
                 fuel_type, tax, mpg, engine_size
        HAVING COUNT(*) > 1
    )
    SELECT COUNT(*) AS identical_record_groups,
           SUM(record_count) AS rows_in_identical_groups,
           SUM(record_count - 1) AS excess_identical_rows,
           ROUND(100.0 * SUM(record_count - 1) / (SELECT COUNT(*) FROM used_cars), 2) AS excess_rows_percentage,
           MAX(record_count) AS largest_identical_group
    FROM grp
""", conn)
duplicate_summary

In [ ]:
# eyeball the 20 biggest duplicate groups
largest_duplicate_groups = pd.read_sql_query("""
    SELECT manufacturer, model, year, price, transmission, mileage,
           fuel_type, tax, mpg, engine_size, COUNT(*) AS record_count
    FROM used_cars
    GROUP BY manufacturer, model, year, price, transmission, mileage,
             fuel_type, tax, mpg, engine_size
    HAVING COUNT(*) > 1
    ORDER BY record_count DESC, manufacturer, model
    LIMIT 20
""", conn)
largest_duplicate_groups

In [ ]:
# where the dupes sit by make, and how many are at <=1000 miles (nearly-new)
duplicates_by_manufacturer = pd.read_sql_query("""
    WITH totals AS (
        SELECT manufacturer, COUNT(*) AS manufacturer_rows FROM used_cars GROUP BY manufacturer
    ),
    grp AS (
        SELECT manufacturer, model, year, price, transmission, mileage,
               fuel_type, tax, mpg, engine_size, COUNT(*) AS record_count
        FROM used_cars
        GROUP BY manufacturer, model, year, price, transmission, mileage,
                 fuel_type, tax, mpg, engine_size
        HAVING COUNT(*) > 1
    )
    SELECT g.manufacturer, t.manufacturer_rows,
           COUNT(*) AS identical_groups,
           SUM(g.record_count) AS rows_in_identical_groups,
           SUM(g.record_count - 1) AS excess_identical_rows,
           ROUND(100.0 * SUM(g.record_count - 1) / t.manufacturer_rows, 2) AS excess_percentage_within_manufacturer,
           SUM(CASE WHEN g.mileage <= 1000 THEN g.record_count - 1 ELSE 0 END) AS excess_rows_at_1000_miles_or_less
    FROM grp g
    JOIN totals t ON g.manufacturer = t.manufacturer
    GROUP BY g.manufacturer, t.manufacturer_rows
    ORDER BY excess_identical_rows DESC
""", conn)
duplicates_by_manufacturer

In [ ]:
# collapse the exact dupes -> one row each, keep a count of how many it stood for
conn.executescript("""
    DROP TABLE IF EXISTS used_cars_deduplicated;
    CREATE TABLE used_cars_deduplicated AS
    SELECT manufacturer, model, year, price, transmission, mileage,
           fuel_type, tax, mpg, engine_size,
           MIN(source_file) AS source_file, COUNT(*) AS exact_record_count
    FROM used_cars
    GROUP BY manufacturer, model, year, price, transmission, mileage,
             fuel_type, tax, mpg, engine_size;
""")

pd.read_sql_query("""
    SELECT COUNT(*) AS deduplicated_rows,
           SUM(exact_record_count) AS original_rows_represented,
           SUM(CASE WHEN exact_record_count > 1 THEN 1 ELSE 0 END) AS repeated_record_groups,
           SUM(exact_record_count - 1) AS collapsed_excess_rows,
           MAX(exact_record_count) AS largest_identical_group
    FROM used_cars_deduplicated
""", conn)

## 2. Range checks on the numeric fields

In [ ]:
numeric_range_audit = pd.read_sql_query("""
    SELECT 'year' AS field_name, COUNT(*) AS record_count, COUNT(DISTINCT year) AS distinct_values,
           MIN(year) AS minimum_value, ROUND(AVG(year),2) AS average_value, MAX(year) AS maximum_value,
           SUM(CASE WHEN year=0 THEN 1 ELSE 0 END) AS zero_records,
           SUM(CASE WHEN year<0 THEN 1 ELSE 0 END) AS negative_records
    FROM used_cars_deduplicated
    UNION ALL
    SELECT 'price', COUNT(*), COUNT(DISTINCT price), MIN(price), ROUND(AVG(price),2), MAX(price),
           SUM(CASE WHEN price=0 THEN 1 ELSE 0 END), SUM(CASE WHEN price<0 THEN 1 ELSE 0 END)
    FROM used_cars_deduplicated
    UNION ALL
    SELECT 'mileage', COUNT(*), COUNT(DISTINCT mileage), MIN(mileage), ROUND(AVG(mileage),2), MAX(mileage),
           SUM(CASE WHEN mileage=0 THEN 1 ELSE 0 END), SUM(CASE WHEN mileage<0 THEN 1 ELSE 0 END)
    FROM used_cars_deduplicated
    UNION ALL
    SELECT 'tax', COUNT(*), COUNT(DISTINCT tax), MIN(tax), ROUND(AVG(tax),2), MAX(tax),
           SUM(CASE WHEN tax=0 THEN 1 ELSE 0 END), SUM(CASE WHEN tax<0 THEN 1 ELSE 0 END)
    FROM used_cars_deduplicated
    UNION ALL
    SELECT 'mpg', COUNT(*), COUNT(DISTINCT mpg), MIN(mpg), ROUND(AVG(mpg),2), MAX(mpg),
           SUM(CASE WHEN mpg=0 THEN 1 ELSE 0 END), SUM(CASE WHEN mpg<0 THEN 1 ELSE 0 END)
    FROM used_cars_deduplicated
    UNION ALL
    SELECT 'engine_size', COUNT(*), COUNT(DISTINCT engine_size), MIN(engine_size), ROUND(AVG(engine_size),2), MAX(engine_size),
           SUM(CASE WHEN engine_size=0 THEN 1 ELSE 0 END), SUM(CASE WHEN engine_size<0 THEN 1 ELSE 0 END)
    FROM used_cars_deduplicated
""", conn)
numeric_range_audit

In [ ]:
year_distribution = pd.read_sql_query("""
    SELECT year, COUNT(*) AS unique_records, SUM(exact_record_count) AS source_records,
           COUNT(DISTINCT manufacturer) AS manufacturer_count
    FROM used_cars_deduplicated
    GROUP BY year ORDER BY year DESC
""", conn)
year_distribution

In [ ]:
# anything after 2020 is off for this snapshot
future_year_records = pd.read_sql_query("""
    SELECT manufacturer, model, year, price, transmission, mileage, fuel_type,
           tax, mpg, engine_size, source_file, exact_record_count
    FROM used_cars_deduplicated
    WHERE year > 2020
    ORDER BY year DESC, manufacturer, model
""", conn)
future_year_records

In [ ]:
# engine_size = 0 -> mostly electrics / missing
zero_engine_summary = pd.read_sql_query("""
    SELECT manufacturer, fuel_type, COUNT(*) AS unique_records,
           SUM(exact_record_count) AS source_records, COUNT(DISTINCT model) AS model_count,
           MIN(year) AS earliest_year, MAX(year) AS latest_year,
           MIN(mpg) AS minimum_mpg, MAX(mpg) AS maximum_mpg
    FROM used_cars_deduplicated
    WHERE engine_size = 0
    GROUP BY manufacturer, fuel_type
    ORDER BY source_records DESC, manufacturer, fuel_type
""", conn)
zero_engine_summary

In [ ]:
extreme_mpg = pd.read_sql_query("""
    SELECT manufacturer, model, fuel_type, mpg,
           MIN(year) AS earliest_year, MAX(year) AS latest_year,
           MIN(engine_size) AS minimum_engine_size, MAX(engine_size) AS maximum_engine_size,
           COUNT(*) AS unique_records, SUM(exact_record_count) AS source_records
    FROM used_cars_deduplicated
    WHERE mpg < 10 OR mpg > 150
    GROUP BY manufacturer, model, fuel_type, mpg
    ORDER BY mpg DESC, manufacturer, model
""", conn)
extreme_mpg

In [ ]:
# cheap / six-figure / very-high-mileage / pre-1980
remaining_extremes = pd.read_sql_query("""
    SELECT 'price_at_or_below_500' AS review_flag, COUNT(*) AS unique_records,
           SUM(exact_record_count) AS source_records, MIN(year) AS earliest_year, MAX(year) AS latest_year,
           MIN(price) AS minimum_price, MAX(price) AS maximum_price,
           MIN(mileage) AS minimum_mileage, MAX(mileage) AS maximum_mileage
    FROM used_cars_deduplicated WHERE price <= 500
    UNION ALL
    SELECT 'price_at_or_above_100000', COUNT(*), SUM(exact_record_count), MIN(year), MAX(year),
           MIN(price), MAX(price), MIN(mileage), MAX(mileage)
    FROM used_cars_deduplicated WHERE price >= 100000
    UNION ALL
    SELECT 'mileage_at_or_above_200000', COUNT(*), SUM(exact_record_count), MIN(year), MAX(year),
           MIN(price), MAX(price), MIN(mileage), MAX(mileage)
    FROM used_cars_deduplicated WHERE mileage >= 200000
    UNION ALL
    SELECT 'model_year_before_1980', COUNT(*), SUM(exact_record_count), MIN(year), MAX(year),
           MIN(price), MAX(price), MIN(mileage), MAX(mileage)
    FROM used_cars_deduplicated WHERE year < 1980
""", conn)
remaining_extremes

In [ ]:
extreme_record_details = pd.read_sql_query("""
    SELECT CASE WHEN year < 1980 THEN 'model_year_before_1980'
                WHEN price <= 500 THEN 'price_at_or_below_500'
                WHEN mileage >= 200000 THEN 'mileage_at_or_above_200000' END AS review_flag,
           manufacturer, model, year, price, transmission, mileage, fuel_type,
           tax, mpg, engine_size, source_file, exact_record_count
    FROM used_cars_deduplicated
    WHERE year < 1980 OR price <= 500 OR mileage >= 200000
    ORDER BY review_flag, year, manufacturer, model
""", conn)
extreme_record_details

## 3. Text hygiene & categories

In [ ]:
text_hygiene = pd.read_sql_query("""
    SELECT 'manufacturer' AS field_name, COUNT(*) AS total_records,
           COUNT(DISTINCT manufacturer) AS raw_distinct_values,
           COUNT(DISTINCT LOWER(TRIM(manufacturer))) AS normalized_distinct_values,
           SUM(CASE WHEN manufacturer <> TRIM(manufacturer) THEN 1 ELSE 0 END) AS records_with_outer_spaces
    FROM used_cars_deduplicated
    UNION ALL
    SELECT 'model', COUNT(*), COUNT(DISTINCT model), COUNT(DISTINCT LOWER(TRIM(model))),
           SUM(CASE WHEN model <> TRIM(model) THEN 1 ELSE 0 END)
    FROM used_cars_deduplicated
    UNION ALL
    SELECT 'transmission', COUNT(*), COUNT(DISTINCT transmission), COUNT(DISTINCT LOWER(TRIM(transmission))),
           SUM(CASE WHEN transmission <> TRIM(transmission) THEN 1 ELSE 0 END)
    FROM used_cars_deduplicated
    UNION ALL
    SELECT 'fuel_type', COUNT(*), COUNT(DISTINCT fuel_type), COUNT(DISTINCT LOWER(TRIM(fuel_type))),
           SUM(CASE WHEN fuel_type <> TRIM(fuel_type) THEN 1 ELSE 0 END)
    FROM used_cars_deduplicated
    UNION ALL
    SELECT 'source_file', COUNT(*), COUNT(DISTINCT source_file), COUNT(DISTINCT LOWER(TRIM(source_file))),
           SUM(CASE WHEN source_file <> TRIM(source_file) THEN 1 ELSE 0 END)
    FROM used_cars_deduplicated
""", conn)
text_hygiene

In [ ]:
category_composition = pd.read_sql_query("""
    SELECT 'manufacturer' AS field_name, manufacturer AS category_value,
           COUNT(*) AS unique_records, SUM(exact_record_count) AS source_records
    FROM used_cars_deduplicated GROUP BY manufacturer
    UNION ALL
    SELECT 'transmission', transmission, COUNT(*), SUM(exact_record_count)
    FROM used_cars_deduplicated GROUP BY transmission
    UNION ALL
    SELECT 'fuel_type', fuel_type, COUNT(*), SUM(exact_record_count)
    FROM used_cars_deduplicated GROUP BY fuel_type
    ORDER BY field_name, source_records DESC
""", conn)
category_composition

In [ ]:
# what's actually in the thin fuel buckets (Other / Electric)
rare_fuel_categories = pd.read_sql_query("""
    SELECT fuel_type, manufacturer, TRIM(model) AS model,
           COUNT(*) AS unique_records, SUM(exact_record_count) AS source_records,
           MIN(year) AS earliest_year, MAX(year) AS latest_year,
           MIN(mpg) AS minimum_mpg, MAX(mpg) AS maximum_mpg
    FROM used_cars_deduplicated
    WHERE fuel_type IN ('Other', 'Electric')
    GROUP BY fuel_type, manufacturer, TRIM(model)
    ORDER BY fuel_type, source_records DESC, manufacturer, model
""", conn)
rare_fuel_categories

In [ ]:
other_transmission_records = pd.read_sql_query("""
    SELECT manufacturer, TRIM(model) AS model, year, price, transmission,
           mileage, fuel_type, tax, mpg, engine_size, exact_record_count
    FROM used_cars_deduplicated
    WHERE transmission = 'Other'
    ORDER BY manufacturer, model, year
""", conn)
other_transmission_records

In [ ]:
source_mapping = pd.read_sql_query("""
    SELECT source_file, COUNT(DISTINCT manufacturer) AS manufacturer_count,
           GROUP_CONCAT(DISTINCT manufacturer) AS manufacturers,
           COUNT(*) AS unique_records, SUM(exact_record_count) AS source_records
    FROM used_cars_deduplicated GROUP BY source_file ORDER BY source_file
""", conn)
source_mapping

In [ ]:
shared_models = pd.read_sql_query("""
    SELECT TRIM(model) AS model, COUNT(DISTINCT manufacturer) AS manufacturer_count,
           GROUP_CONCAT(DISTINCT manufacturer) AS manufacturers,
           COUNT(*) AS unique_records, SUM(exact_record_count) AS source_records
    FROM used_cars_deduplicated
    GROUP BY TRIM(model)
    HAVING COUNT(DISTINCT manufacturer) > 1
    ORDER BY manufacturer_count DESC, model
""", conn)
shared_models

## 4. Build the clean table

Keep raw + cleaned columns side by side with a few flags, restrict to
1980-2020, and treat age as 2020 - year (snapshot year).

In [ ]:
conn.executescript("""
    DROP TABLE IF EXISTS used_cars_clean;
    CREATE TABLE used_cars_clean AS
    SELECT manufacturer,
           TRIM(model) AS model,
           manufacturer || ' | ' || TRIM(model) AS manufacturer_model,
           year AS model_year,
           2020 - year AS approximate_vehicle_age,
           price AS listed_price_gbp,
           transmission AS transmission_raw,
           CASE WHEN transmission = 'Other' THEN NULL ELSE transmission END AS transmission_clean,
           mileage AS mileage_miles,
           fuel_type AS fuel_type_raw,
           CASE WHEN fuel_type IN ('Electric','Other') THEN 'Other / unclear' ELSE fuel_type END AS fuel_type_analysis,
           tax AS annual_tax_gbp,
           mpg AS mpg_raw,
           CASE WHEN mpg < 10 THEN NULL ELSE mpg END AS mpg_clean,
           CASE WHEN mpg > 150 THEN 1 ELSE 0 END AS high_mpg_flag,
           CASE WHEN mpg < 10 THEN 1 ELSE 0 END AS invalid_mpg_flag,
           engine_size AS engine_size_raw,
           CASE WHEN engine_size = 0 THEN NULL ELSE engine_size END AS engine_size_clean,
           CASE WHEN engine_size = 0 THEN 1 ELSE 0 END AS engine_size_unavailable_flag,
           CASE WHEN transmission = 'Other' THEN 1 ELSE 0 END AS transmission_unavailable_flag,
           CASE WHEN fuel_type IN ('Electric','Other') THEN 1 ELSE 0 END AS unclear_fuel_type_flag,
           source_file, exact_record_count
    FROM used_cars_deduplicated
    WHERE year BETWEEN 1980 AND 2020;
""")

pd.read_sql_query("""
    SELECT COUNT(*) AS clean_unique_records, SUM(exact_record_count) AS source_records_represented,
           COUNT(DISTINCT manufacturer) AS manufacturers, COUNT(DISTINCT model) AS models,
           COUNT(DISTINCT manufacturer_model) AS manufacturer_models,
           MIN(model_year) AS earliest_model_year, MAX(model_year) AS latest_model_year,
           MIN(approximate_vehicle_age) AS minimum_approximate_age, MAX(approximate_vehicle_age) AS maximum_approximate_age,
           SUM(invalid_mpg_flag) AS invalid_mpg_records, SUM(high_mpg_flag) AS high_mpg_records,
           SUM(engine_size_unavailable_flag) AS unavailable_engine_size_records,
           SUM(transmission_unavailable_flag) AS unavailable_transmission_records,
           SUM(unclear_fuel_type_flag) AS unclear_fuel_type_records
    FROM used_cars_clean
""", conn)

In [ ]:
# nothing should go missing staging -> dedup -> clean
final_reconciliation = pd.read_sql_query("""
    WITH c AS (
        SELECT (SELECT COUNT(*) FROM used_cars) AS staging_rows,
               (SELECT COUNT(*) FROM used_cars_deduplicated) AS deduplicated_unique_records,
               (SELECT COUNT(*) FROM used_cars_clean) AS clean_unique_records,
               (SELECT SUM(exact_record_count) FROM used_cars_clean) AS clean_source_records,
               (SELECT SUM(exact_record_count) FROM used_cars_deduplicated WHERE year NOT BETWEEN 1980 AND 2020) AS invalid_year_source_records,
               (SELECT COUNT(*) FROM used_cars_deduplicated WHERE year NOT BETWEEN 1980 AND 2020) AS invalid_year_unique_records
    )
    SELECT staging_rows, deduplicated_unique_records,
           staging_rows - deduplicated_unique_records AS collapsed_excess_duplicates,
           clean_unique_records, invalid_year_unique_records, invalid_year_source_records, clean_source_records,
           staging_rows - clean_source_records - invalid_year_source_records AS reconciliation_difference
    FROM c
""", conn)
final_reconciliation

In [ ]:
# distribution of the key numeric fields on the clean table
numeric_audit_data = pd.read_sql_query("""
    SELECT listed_price_gbp, mileage_miles, annual_tax_gbp, mpg_clean,
           engine_size_clean, approximate_vehicle_age
    FROM used_cars_clean
""", conn)

qs = [0.0, 0.001, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999, 1.0]
labels = ["minimum", "p0_1", "p1", "p5", "p25", "median", "p75", "p95", "p99", "p99_9", "maximum"]

quantile_audit = numeric_audit_data.quantile(qs).T
quantile_audit.columns = labels
quantile_audit.insert(0, "non_missing_records", numeric_audit_data.notna().sum())
quantile_audit.insert(1, "missing_records", numeric_audit_data.isna().sum())
quantile_audit = quantile_audit.reset_index().rename(columns={"index": "field_name"}).round(2)
quantile_audit

### The £100k+ listings
Most of the six-figure prices are genuine (S-Class, X5 etc.) but a couple
sit on cheap models and are clearly data entry errors.

In [ ]:
high_price_model_summary = pd.read_sql_query("""
    SELECT manufacturer, model, COUNT(*) AS unique_records,
           MIN(model_year) AS earliest_year, MAX(model_year) AS latest_year,
           MIN(listed_price_gbp) AS minimum_price, MAX(listed_price_gbp) AS maximum_price,
           MIN(mileage_miles) AS minimum_mileage, MAX(mileage_miles) AS maximum_mileage
    FROM used_cars_clean
    WHERE listed_price_gbp >= 100000
    GROUP BY manufacturer, model
    ORDER BY unique_records DESC, maximum_price DESC
""", conn)
high_price_model_summary

In [ ]:
# the two that jump out - A Class and 2 Series at 100k+
questionable_high_price = pd.read_sql_query("""
    SELECT manufacturer, model, model_year, listed_price_gbp, transmission_raw,
           mileage_miles, fuel_type_raw, annual_tax_gbp, mpg_raw, engine_size_raw,
           source_file, exact_record_count
    FROM used_cars_clean
    WHERE (manufacturer = 'Mercedes-Benz' AND model = 'A Class' AND listed_price_gbp >= 100000)
       OR (manufacturer = 'BMW' AND model = '2 Series' AND listed_price_gbp >= 100000)
    ORDER BY manufacturer, model, listed_price_gbp
""", conn)
questionable_high_price

In [ ]:
# A Class split by engine size - the 4.0 rows are the mislabelled ones
a_class_engine_comparison = pd.read_sql_query("""
    SELECT engine_size_raw, COUNT(*) AS unique_records,
           MIN(model_year) AS earliest_year, MAX(model_year) AS latest_year,
           ROUND(AVG(listed_price_gbp),2) AS average_price,
           MIN(listed_price_gbp) AS minimum_price, MAX(listed_price_gbp) AS maximum_price,
           MIN(mpg_raw) AS minimum_mpg, MAX(mpg_raw) AS maximum_mpg
    FROM used_cars_clean
    WHERE manufacturer = 'Mercedes-Benz' AND model = 'A Class'
    GROUP BY engine_size_raw ORDER BY engine_size_raw
""", conn)
a_class_engine_comparison

In [ ]:
# comparable 2 Series diesels - shows the 123456 price is nonsense
bmw_2_series_comparison = pd.read_sql_query("""
    SELECT model_year, listed_price_gbp, mileage_miles, transmission_raw,
           fuel_type_raw, engine_size_raw, annual_tax_gbp, mpg_raw
    FROM used_cars_clean
    WHERE manufacturer = 'BMW' AND model = '2 Series'
      AND model_year BETWEEN 2014 AND 2016 AND fuel_type_raw = 'Diesel' AND engine_size_raw = 2.0
    ORDER BY listed_price_gbp DESC LIMIT 15
""", conn)
bmw_2_series_comparison

In [ ]:
large_engine_summary = pd.read_sql_query("""
    SELECT manufacturer, model, fuel_type_analysis, engine_size_clean,
           COUNT(*) AS unique_records, MIN(model_year) AS earliest_year, MAX(model_year) AS latest_year,
           MIN(listed_price_gbp) AS minimum_price, MAX(listed_price_gbp) AS maximum_price
    FROM used_cars_clean
    WHERE engine_size_clean > 5
    GROUP BY manufacturer, model, fuel_type_analysis, engine_size_clean
    ORDER BY engine_size_clean DESC, unique_records DESC
""", conn)
large_engine_summary

In [ ]:
# £0 tax - clusters in newer low-emission cars
zero_tax_structure = pd.read_sql_query("""
    WITH base AS (
        SELECT CASE WHEN model_year <= 2010 THEN '1996-2010'
                    WHEN model_year <= 2016 THEN '2011-2016'
                    ELSE '2017-2020' END AS model_year_band,
               fuel_type_analysis, manufacturer_model, annual_tax_gbp, mpg_clean
        FROM used_cars_clean
    )
    SELECT model_year_band, fuel_type_analysis, COUNT(*) AS total_unique_records,
           SUM(CASE WHEN annual_tax_gbp = 0 THEN 1 ELSE 0 END) AS zero_tax_records,
           ROUND(100.0 * SUM(CASE WHEN annual_tax_gbp = 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS zero_tax_percentage,
           COUNT(DISTINCT CASE WHEN annual_tax_gbp = 0 THEN manufacturer_model END) AS zero_tax_model_count,
           ROUND(AVG(CASE WHEN annual_tax_gbp = 0 THEN mpg_clean END), 2) AS average_mpg_for_zero_tax
    FROM base
    GROUP BY model_year_band, fuel_type_analysis
    HAVING SUM(CASE WHEN annual_tax_gbp = 0 THEN 1 ELSE 0 END) > 0
    ORDER BY model_year_band, zero_tax_records DESC
""", conn)
zero_tax_structure

### Sample size per model
Excluding the two bad clusters, how much data sits behind each model and
each model x age band.

In [ ]:
model_support_audit = pd.read_sql_query("""
    WITH eligible AS (
        SELECT * FROM used_cars_clean
        WHERE NOT (manufacturer = 'Mercedes-Benz' AND model = 'A Class' AND engine_size_raw = 4.0)
          AND NOT (manufacturer = 'BMW' AND model = '2 Series' AND model_year = 2015
                   AND listed_price_gbp = 123456 AND mileage_miles = 33419)
    ),
    support AS (SELECT manufacturer_model, COUNT(*) AS unique_records FROM eligible GROUP BY manufacturer_model),
    banded AS (
        SELECT unique_records,
               CASE WHEN unique_records < 25 THEN '01: fewer than 25'
                    WHEN unique_records < 50 THEN '02: 25-49'
                    WHEN unique_records < 100 THEN '03: 50-99'
                    WHEN unique_records < 250 THEN '04: 100-249'
                    WHEN unique_records < 500 THEN '05: 250-499'
                    ELSE '06: 500 or more' END AS support_band
        FROM support
    )
    SELECT support_band, COUNT(*) AS manufacturer_model_count, SUM(unique_records) AS records_in_band,
           MIN(unique_records) AS smallest_model_group, MAX(unique_records) AS largest_model_group,
           ROUND(100.0 * SUM(unique_records) / (SELECT SUM(unique_records) FROM support), 2) AS percentage_of_eligible_records
    FROM banded GROUP BY support_band ORDER BY support_band
""", conn)
model_support_audit

In [ ]:
model_age_support = pd.read_sql_query("""
    WITH eligible AS (
        SELECT *,
               CASE WHEN approximate_vehicle_age <= 2 THEN '0-2 years'
                    WHEN approximate_vehicle_age <= 5 THEN '3-5 years'
                    WHEN approximate_vehicle_age <= 10 THEN '6-10 years'
                    ELSE '11+ years' END AS age_band
        FROM used_cars_clean
        WHERE NOT (manufacturer = 'Mercedes-Benz' AND model = 'A Class' AND engine_size_raw = 4.0)
          AND NOT (manufacturer = 'BMW' AND model = '2 Series' AND model_year = 2015
                   AND listed_price_gbp = 123456 AND mileage_miles = 33419)
    ),
    seg AS (SELECT manufacturer_model, age_band, COUNT(*) AS segment_records FROM eligible GROUP BY manufacturer_model, age_band),
    banded AS (
        SELECT segment_records,
               CASE WHEN segment_records < 5 THEN '01: fewer than 5'
                    WHEN segment_records < 10 THEN '02: 5-9'
                    WHEN segment_records < 25 THEN '03: 10-24'
                    WHEN segment_records < 50 THEN '04: 25-49'
                    WHEN segment_records < 100 THEN '05: 50-99'
                    ELSE '06: 100 or more' END AS support_band
        FROM seg
    )
    SELECT support_band, COUNT(*) AS model_age_segments, SUM(segment_records) AS records_in_segments,
           MIN(segment_records) AS smallest_segment, MAX(segment_records) AS largest_segment,
           ROUND(100.0 * SUM(segment_records) / (SELECT SUM(segment_records) FROM seg), 2) AS percentage_of_eligible_records
    FROM banded GROUP BY support_band ORDER BY support_band
""", conn)
model_age_support

## 5. Analysis table with quality flags

Flag the mislabelled A Class and the bogus 2 Series price, then rebuild the
clean table with those flags and take the rows that pass both.

In [ ]:
conn.executescript("""
    DROP TABLE IF EXISTS used_cars_analysis;
    DROP TABLE IF EXISTS used_cars_clean;
    CREATE TABLE used_cars_clean AS
    WITH cleaned AS (
        SELECT manufacturer,
               TRIM(model) AS model,
               manufacturer || ' | ' || TRIM(model) AS manufacturer_model,
               year AS model_year,
               2020 - year AS approximate_vehicle_age,
               price AS listed_price_gbp,
               transmission AS transmission_raw,
               CASE WHEN transmission = 'Other' THEN NULL ELSE transmission END AS transmission_clean,
               mileage AS mileage_miles,
               fuel_type AS fuel_type_raw,
               CASE WHEN fuel_type IN ('Electric','Other') THEN 'Other / unclear' ELSE fuel_type END AS fuel_type_analysis,
               tax AS annual_tax_gbp,
               mpg AS mpg_raw,
               CASE WHEN mpg < 10 THEN NULL ELSE mpg END AS mpg_clean,
               CASE WHEN mpg > 150 THEN 1 ELSE 0 END AS high_mpg_flag,
               CASE WHEN mpg < 10 THEN 1 ELSE 0 END AS invalid_mpg_flag,
               engine_size AS engine_size_raw,
               CASE WHEN engine_size = 0 THEN NULL ELSE engine_size END AS engine_size_clean,
               CASE WHEN engine_size = 0 THEN 1 ELSE 0 END AS engine_size_unavailable_flag,
               CASE WHEN transmission = 'Other' THEN 1 ELSE 0 END AS transmission_unavailable_flag,
               CASE WHEN fuel_type IN ('Electric','Other') THEN 1 ELSE 0 END AS unclear_fuel_type_flag,
               source_file, exact_record_count
        FROM used_cars_deduplicated
        WHERE year BETWEEN 1980 AND 2020
    ),
    flagged AS (
        SELECT *,
               CASE WHEN manufacturer = 'Mercedes-Benz' AND model = 'A Class' AND engine_size_raw = 4.0
                    THEN 1 ELSE 0 END AS model_label_unreliable_flag,
               CASE WHEN manufacturer = 'BMW' AND model = '2 Series' AND model_year = 2015
                         AND listed_price_gbp = 123456 AND mileage_miles = 33419
                    THEN 1 ELSE 0 END AS invalid_price_flag
        FROM cleaned
    )
    SELECT *,
           CASE WHEN model_label_unreliable_flag = 1 OR invalid_price_flag = 1 THEN 0 ELSE 1 END AS core_analysis_eligible_flag
    FROM flagged;

    CREATE TABLE used_cars_analysis AS
    SELECT * FROM used_cars_clean WHERE core_analysis_eligible_flag = 1;
""")

pd.read_sql_query("""
    SELECT (SELECT COUNT(*) FROM used_cars_clean) AS clean_audit_records,
           (SELECT SUM(model_label_unreliable_flag) FROM used_cars_clean) AS unreliable_model_records,
           (SELECT SUM(invalid_price_flag) FROM used_cars_clean) AS invalid_price_records,
           (SELECT COUNT(*) FROM used_cars_analysis) AS analysis_eligible_records,
           (SELECT COUNT(DISTINCT manufacturer) FROM used_cars_analysis) AS eligible_manufacturers,
           (SELECT COUNT(DISTINCT manufacturer_model) FROM used_cars_analysis) AS eligible_manufacturer_models
""", conn)

In [ ]:
# same car attributes, different price = probably relistings / scrape dupes.
# how many "fingerprints" carry more than one price?
non_price_fingerprint_summary = pd.read_sql_query("""
    WITH fp AS (
        SELECT manufacturer, model, model_year, transmission_raw, mileage_miles,
               fuel_type_raw, annual_tax_gbp, mpg_raw, engine_size_raw,
               COUNT(*) AS unique_price_records,
               COUNT(DISTINCT listed_price_gbp) AS distinct_prices,
               SUM(exact_record_count) AS source_records,
               MIN(listed_price_gbp) AS minimum_price, MAX(listed_price_gbp) AS maximum_price
        FROM used_cars_clean
        GROUP BY manufacturer, model, model_year, transmission_raw, mileage_miles,
                 fuel_type_raw, annual_tax_gbp, mpg_raw, engine_size_raw
        HAVING COUNT(DISTINCT listed_price_gbp) > 1
    )
    SELECT COUNT(*) AS fingerprints_with_multiple_prices,
           SUM(unique_price_records) AS unique_records_in_fingerprints,
           SUM(source_records) AS source_records_in_fingerprints,
           SUM(unique_price_records - 1) AS excess_unique_price_records,
           MAX(distinct_prices) AS largest_number_of_distinct_prices,
           ROUND(AVG(maximum_price - minimum_price), 2) AS average_price_range,
           MAX(maximum_price - minimum_price) AS largest_price_range
    FROM fp
""", conn)
non_price_fingerprint_summary

In [ ]:
largest_fingerprint_groups = pd.read_sql_query("""
    WITH fp AS (
        SELECT manufacturer, model, model_year, transmission_raw, mileage_miles,
               fuel_type_raw, annual_tax_gbp, mpg_raw, engine_size_raw,
               COUNT(*) AS unique_price_records,
               COUNT(DISTINCT listed_price_gbp) AS distinct_prices,
               SUM(exact_record_count) AS source_records,
               MIN(listed_price_gbp) AS minimum_price, MAX(listed_price_gbp) AS maximum_price,
               ROUND(AVG(listed_price_gbp),2) AS average_price
        FROM used_cars_clean
        GROUP BY manufacturer, model, model_year, transmission_raw, mileage_miles,
                 fuel_type_raw, annual_tax_gbp, mpg_raw, engine_size_raw
        HAVING COUNT(DISTINCT listed_price_gbp) > 1
    )
    SELECT *, maximum_price - minimum_price AS price_range
    FROM fp ORDER BY distinct_prices DESC, price_range DESC LIMIT 20
""", conn)
largest_fingerprint_groups

In [ ]:
# flag prices 4x above or 1/4 below the model x year median (groups of 25+)
pq = pd.read_sql_query("""
    SELECT manufacturer, model, manufacturer_model, model_year, listed_price_gbp,
           mileage_miles, transmission_raw, fuel_type_raw, engine_size_raw
    FROM used_cars_clean WHERE core_analysis_eligible_flag = 1
""", conn)

g = pq.groupby(["manufacturer_model", "model_year"])["listed_price_gbp"]
pq["group_record_count"] = g.transform("size")
pq["group_median_price"] = g.transform("median")
pq["price_to_group_median"] = pq["listed_price_gbp"] / pq["group_median_price"]

price_quality_candidates = pq[
    (pq["group_record_count"] >= 25)
    & ((pq["price_to_group_median"] >= 4) | (pq["price_to_group_median"] <= 0.25))
].copy()
price_quality_candidates["deviation_multiple"] = price_quality_candidates["price_to_group_median"].where(
    price_quality_candidates["price_to_group_median"] >= 1,
    1 / price_quality_candidates["price_to_group_median"]
)
price_quality_candidates = price_quality_candidates.sort_values("deviation_multiple", ascending=False).round(2)
price_quality_candidates

In [ ]:
# add the extra bad prices found above, rebuild eligibility, and stamp a
# fingerprint id (same non-price attributes -> same id) for grouping later
conn.executescript("""
    UPDATE used_cars_clean
    SET invalid_price_flag = CASE WHEN
        (manufacturer='BMW' AND model='2 Series' AND model_year=2015 AND listed_price_gbp=123456 AND mileage_miles=33419)
        OR (manufacturer='Hyundai' AND model='I10' AND model_year=2017 AND listed_price_gbp=92000 AND mileage_miles=35460)
        OR (manufacturer='Vauxhall' AND model='Mokka' AND model_year=2016 AND listed_price_gbp=52489 AND mileage_miles=52489)
        OR (manufacturer='Skoda' AND model='Karoq' AND model_year=2019 AND listed_price_gbp=91874 AND mileage_miles=3764)
        THEN 1 ELSE 0 END;

    UPDATE used_cars_clean
    SET core_analysis_eligible_flag = CASE WHEN model_label_unreliable_flag = 1 OR invalid_price_flag = 1 THEN 0 ELSE 1 END;

    DROP TABLE IF EXISTS used_cars_analysis;
    CREATE TABLE used_cars_analysis AS
    SELECT c.*,
           DENSE_RANK() OVER (
               ORDER BY manufacturer, model, model_year, transmission_raw, mileage_miles,
                        fuel_type_raw, annual_tax_gbp, mpg_raw, engine_size_raw
           ) AS non_price_fingerprint_id
    FROM used_cars_clean c
    WHERE core_analysis_eligible_flag = 1;
""")

pd.read_sql_query("""
    SELECT (SELECT SUM(invalid_price_flag) FROM used_cars_clean) AS invalid_price_records,
           (SELECT COUNT(*) FROM used_cars_analysis) AS analysis_eligible_records,
           (SELECT COUNT(DISTINCT non_price_fingerprint_id) FROM used_cars_analysis) AS eligible_fingerprint_groups,
           (SELECT COUNT(DISTINCT manufacturer) FROM used_cars_analysis) AS eligible_manufacturers,
           (SELECT COUNT(DISTINCT manufacturer_model) FROM used_cars_analysis) AS eligible_manufacturer_models
""", conn)

## 6. Portfolio landscape

In [ ]:
portfolio_overview = pd.read_sql_query("""
    WITH ranked AS (
        SELECT listed_price_gbp, ROW_NUMBER() OVER (ORDER BY listed_price_gbp) AS r,
               COUNT(*) OVER () AS n
        FROM used_cars_analysis
    ),
    stats AS (
        SELECT MIN(listed_price_gbp) AS minimum_price,
               MIN(CASE WHEN r >= 0.25 * n THEN listed_price_gbp END) AS p25_price,
               AVG(CASE WHEN r IN ((n+1)/2, (n+2)/2) THEN listed_price_gbp END) AS median_price,
               ROUND(AVG(listed_price_gbp),2) AS average_price,
               MIN(CASE WHEN r >= 0.75 * n THEN listed_price_gbp END) AS p75_price,
               MIN(CASE WHEN r >= 0.95 * n THEN listed_price_gbp END) AS p95_price,
               MAX(listed_price_gbp) AS maximum_price
        FROM ranked
    )
    SELECT (SELECT COUNT(*) FROM used_cars_analysis) AS analysis_records,
           (SELECT SUM(exact_record_count) FROM used_cars_analysis) AS source_records_represented,
           (SELECT COUNT(DISTINCT non_price_fingerprint_id) FROM used_cars_analysis) AS fingerprint_groups,
           (SELECT COUNT(DISTINCT manufacturer) FROM used_cars_analysis) AS manufacturers,
           (SELECT COUNT(DISTINCT manufacturer_model) FROM used_cars_analysis) AS manufacturer_models,
           (SELECT MIN(model_year) FROM used_cars_analysis) AS earliest_model_year,
           (SELECT MAX(model_year) FROM used_cars_analysis) AS latest_model_year,
           minimum_price, p25_price, ROUND(median_price,2) AS median_price,
           average_price, p75_price, p95_price, maximum_price
    FROM stats
""", conn)
portfolio_overview

In [ ]:
manufacturer_landscape = pd.read_sql_query("""
    WITH ranked AS (
        SELECT manufacturer, manufacturer_model, non_price_fingerprint_id, listed_price_gbp,
               ROW_NUMBER() OVER (PARTITION BY manufacturer ORDER BY listed_price_gbp) AS r,
               COUNT(*) OVER (PARTITION BY manufacturer) AS n
        FROM used_cars_analysis
    ),
    stats AS (
        SELECT manufacturer, COUNT(*) AS analysis_records,
               COUNT(DISTINCT non_price_fingerprint_id) AS fingerprint_groups,
               COUNT(DISTINCT manufacturer_model) AS model_count,
               MIN(listed_price_gbp) AS minimum_price,
               MIN(CASE WHEN r >= 0.25 * n THEN listed_price_gbp END) AS p25_price,
               AVG(CASE WHEN r IN ((n+1)/2, (n+2)/2) THEN listed_price_gbp END) AS median_price,
               ROUND(AVG(listed_price_gbp),2) AS average_price,
               MIN(CASE WHEN r >= 0.75 * n THEN listed_price_gbp END) AS p75_price,
               MIN(CASE WHEN r >= 0.95 * n THEN listed_price_gbp END) AS p95_price,
               MAX(listed_price_gbp) AS maximum_price
        FROM ranked GROUP BY manufacturer
    )
    SELECT manufacturer, analysis_records,
           ROUND(100.0 * analysis_records / (SELECT COUNT(*) FROM used_cars_analysis), 2) AS portfolio_record_share_percentage,
           fingerprint_groups, model_count, minimum_price, p25_price,
           ROUND(median_price,2) AS median_price, average_price, p75_price, p95_price, maximum_price
    FROM stats ORDER BY median_price DESC
""", conn)
manufacturer_landscape

In [ ]:
top_model_landscape = pd.read_sql_query("""
    WITH ranked AS (
        SELECT manufacturer, model, manufacturer_model, non_price_fingerprint_id, listed_price_gbp,
               ROW_NUMBER() OVER (PARTITION BY manufacturer_model ORDER BY listed_price_gbp) AS r,
               COUNT(*) OVER (PARTITION BY manufacturer_model) AS n
        FROM used_cars_analysis
    ),
    stats AS (
        SELECT manufacturer, model, manufacturer_model, COUNT(*) AS analysis_records,
               COUNT(DISTINCT non_price_fingerprint_id) AS fingerprint_groups,
               MIN(model_year) AS earliest_model_year, MAX(model_year) AS latest_model_year,
               MIN(CASE WHEN r >= 0.25 * n THEN listed_price_gbp END) AS p25_price,
               AVG(CASE WHEN r IN ((n+1)/2, (n+2)/2) THEN listed_price_gbp END) AS median_price,
               MIN(CASE WHEN r >= 0.75 * n THEN listed_price_gbp END) AS p75_price
        FROM ranked GROUP BY manufacturer, model, manufacturer_model
    )
    SELECT manufacturer, model, analysis_records,
           ROUND(100.0 * analysis_records / (SELECT COUNT(*) FROM used_cars_analysis), 2) AS portfolio_record_share_percentage,
           fingerprint_groups, earliest_model_year, latest_model_year, p25_price,
           ROUND(median_price,2) AS median_price, p75_price
    FROM stats ORDER BY analysis_records DESC LIMIT 15
""", conn)
top_model_landscape

In [ ]:
category_landscape = pd.read_sql_query("""
    WITH recs AS (
        SELECT 'Fuel type' AS category_dimension, fuel_type_analysis AS category_value, listed_price_gbp, non_price_fingerprint_id
        FROM used_cars_analysis
        UNION ALL
        SELECT 'Transmission', COALESCE(transmission_clean,'Unavailable'), listed_price_gbp, non_price_fingerprint_id
        FROM used_cars_analysis
    ),
    ranked AS (
        SELECT category_dimension, category_value, listed_price_gbp, non_price_fingerprint_id,
               ROW_NUMBER() OVER (PARTITION BY category_dimension, category_value ORDER BY listed_price_gbp) AS r,
               COUNT(*) OVER (PARTITION BY category_dimension, category_value) AS n
        FROM recs
    ),
    stats AS (
        SELECT category_dimension, category_value, COUNT(*) AS analysis_records,
               COUNT(DISTINCT non_price_fingerprint_id) AS fingerprint_groups,
               MIN(CASE WHEN r >= 0.25 * n THEN listed_price_gbp END) AS p25_price,
               AVG(CASE WHEN r IN ((n+1)/2, (n+2)/2) THEN listed_price_gbp END) AS median_price,
               ROUND(AVG(listed_price_gbp),2) AS average_price,
               MIN(CASE WHEN r >= 0.75 * n THEN listed_price_gbp END) AS p75_price
        FROM ranked GROUP BY category_dimension, category_value
    )
    SELECT category_dimension, category_value, analysis_records,
           ROUND(100.0 * analysis_records / (SELECT COUNT(*) FROM used_cars_analysis), 2) AS portfolio_share_percentage,
           fingerprint_groups, p25_price, ROUND(median_price,2) AS median_price, average_price, p75_price
    FROM stats ORDER BY category_dimension, median_price DESC
""", conn)
category_landscape

## 7. Train/test split

Split on the fingerprint id, not the row, so relistings of the same car
can't leak across the split. 20% held out, stratified by manufacturer.

In [ ]:
fp_pop = pd.read_sql_query("""
    SELECT non_price_fingerprint_id, manufacturer, COUNT(*) AS records_in_fingerprint
    FROM used_cars_analysis
    GROUP BY non_price_fingerprint_id, manufacturer
""", conn)

test_fps = fp_pop.groupby("manufacturer", group_keys=False).sample(frac=0.20, random_state=42)["non_price_fingerprint_id"]
fp_pop["data_split"] = np.where(fp_pop["non_price_fingerprint_id"].isin(test_fps), "test", "train")

fp_pop[["non_price_fingerprint_id", "data_split"]].to_sql("fingerprint_split_map", conn, index=False, if_exists="replace")

conn.executescript("""
    DROP TABLE IF EXISTS used_cars_analysis_split;
    CREATE TABLE used_cars_analysis_split AS
    SELECT a.*, s.data_split
    FROM used_cars_analysis a
    JOIN fingerprint_split_map s ON a.non_price_fingerprint_id = s.non_price_fingerprint_id;
""")

split_structure = pd.read_sql_query("""
    WITH overlap AS (
        SELECT non_price_fingerprint_id FROM used_cars_analysis_split
        GROUP BY non_price_fingerprint_id HAVING COUNT(DISTINCT data_split) > 1
    )
    SELECT data_split, COUNT(*) AS analysis_records,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM used_cars_analysis_split), 2) AS record_percentage,
           COUNT(DISTINCT non_price_fingerprint_id) AS fingerprint_groups,
           COUNT(DISTINCT manufacturer) AS manufacturers,
           COUNT(DISTINCT manufacturer_model) AS manufacturer_models,
           (SELECT COUNT(*) FROM overlap) AS fingerprints_in_both_splits
    FROM used_cars_analysis_split GROUP BY data_split ORDER BY data_split
""", conn)
split_structure

In [ ]:
# check the two halves look alike on price/age/mileage
sb = pd.read_sql_query("""
    SELECT data_split, non_price_fingerprint_id, listed_price_gbp, approximate_vehicle_age, mileage_miles
    FROM used_cars_analysis_split
""", conn)

overall_split_balance = sb.groupby("data_split").agg(
    analysis_records=("listed_price_gbp", "size"),
    fingerprint_groups=("non_price_fingerprint_id", "nunique"),
    average_price=("listed_price_gbp", "mean"),
    median_price=("listed_price_gbp", "median"),
    p25_price=("listed_price_gbp", lambda x: x.quantile(0.25)),
    p75_price=("listed_price_gbp", lambda x: x.quantile(0.75)),
    median_approximate_age=("approximate_vehicle_age", "median"),
    median_mileage=("mileage_miles", "median"),
).reset_index().round(2)
overall_split_balance

In [ ]:
categorical_split_balance = pd.read_sql_query("""
    WITH recs AS (
        SELECT data_split, 'Manufacturer' AS d, manufacturer AS v FROM used_cars_analysis_split
        UNION ALL SELECT data_split, 'Fuel type', fuel_type_analysis FROM used_cars_analysis_split
        UNION ALL SELECT data_split, 'Transmission', COALESCE(transmission_clean,'Unavailable') FROM used_cars_analysis_split
    ),
    counts AS (
        SELECT d AS category_dimension, v AS category_value,
               SUM(CASE WHEN data_split='train' THEN 1 ELSE 0 END) AS train_records,
               SUM(CASE WHEN data_split='test' THEN 1 ELSE 0 END) AS test_records
        FROM recs GROUP BY d, v
    )
    SELECT category_dimension, category_value, train_records, test_records,
           ROUND(100.0 * train_records / (SELECT COUNT(*) FROM used_cars_analysis_split WHERE data_split='train'), 2) AS train_share_percentage,
           ROUND(100.0 * test_records / (SELECT COUNT(*) FROM used_cars_analysis_split WHERE data_split='test'), 2) AS test_share_percentage,
           ROUND(ABS(
               100.0 * train_records / (SELECT COUNT(*) FROM used_cars_analysis_split WHERE data_split='train')
               - 100.0 * test_records / (SELECT COUNT(*) FROM used_cars_analysis_split WHERE data_split='test')
           ), 2) AS absolute_percentage_point_difference
    FROM counts ORDER BY category_dimension, category_value
""", conn)
categorical_split_balance

## 8. Baseline: hierarchical comparables

Train-only median price at five levels of granularity (>=25 fingerprints).
At predict time, walk down the hierarchy until something matches.

In [ ]:
conn.executescript("""
    DROP TABLE IF EXISTS comparable_medians_train;
    CREATE TABLE comparable_medians_train AS
    WITH base AS (
        SELECT manufacturer, manufacturer_model, non_price_fingerprint_id, listed_price_gbp,
               CASE WHEN approximate_vehicle_age <= 2 THEN '0-2 years'
                    WHEN approximate_vehicle_age <= 5 THEN '3-5 years'
                    WHEN approximate_vehicle_age <= 10 THEN '6-10 years'
                    ELSE '11+ years' END AS age_band
        FROM used_cars_analysis_split WHERE data_split = 'train'
    ),
    rows_ AS (
        SELECT '01_model_age' AS segment_level, manufacturer_model || ' || ' || age_band AS segment_key, non_price_fingerprint_id, listed_price_gbp FROM base
        UNION ALL SELECT '02_model', manufacturer_model, non_price_fingerprint_id, listed_price_gbp FROM base
        UNION ALL SELECT '03_manufacturer_age', manufacturer || ' || ' || age_band, non_price_fingerprint_id, listed_price_gbp FROM base
        UNION ALL SELECT '04_manufacturer', manufacturer, non_price_fingerprint_id, listed_price_gbp FROM base
        UNION ALL SELECT '05_overall', 'ALL', non_price_fingerprint_id, listed_price_gbp FROM base
    ),
    support AS (
        SELECT segment_level, segment_key, COUNT(*) AS record_support,
               COUNT(DISTINCT non_price_fingerprint_id) AS fingerprint_support
        FROM rows_ GROUP BY segment_level, segment_key
    ),
    ranked AS (
        SELECT segment_level, segment_key, listed_price_gbp,
               ROW_NUMBER() OVER (PARTITION BY segment_level, segment_key ORDER BY listed_price_gbp) AS r,
               COUNT(*) OVER (PARTITION BY segment_level, segment_key) AS n
        FROM rows_
    ),
    medians AS (
        SELECT r.segment_level, r.segment_key,
               MAX(s.record_support) AS record_support, MAX(s.fingerprint_support) AS fingerprint_support,
               AVG(CASE WHEN r.r IN ((r.n+1)/2, (r.n+2)/2) THEN r.listed_price_gbp END) AS comparable_median_price
        FROM ranked r JOIN support s ON r.segment_level = s.segment_level AND r.segment_key = s.segment_key
        GROUP BY r.segment_level, r.segment_key
    )
    SELECT segment_level, segment_key, record_support, fingerprint_support,
           ROUND(comparable_median_price, 2) AS comparable_median_price
    FROM medians WHERE fingerprint_support >= 25;
""")

pd.read_sql_query("""
    SELECT segment_level, COUNT(*) AS eligible_segments,
           MIN(fingerprint_support) AS minimum_fingerprint_support,
           ROUND(AVG(fingerprint_support),2) AS average_fingerprint_support,
           MAX(fingerprint_support) AS maximum_fingerprint_support,
           MIN(comparable_median_price) AS minimum_segment_median,
           MAX(comparable_median_price) AS maximum_segment_median
    FROM comparable_medians_train GROUP BY segment_level ORDER BY segment_level
""", conn)

In [ ]:
conn.executescript("""
    DROP TABLE IF EXISTS test_comparable_predictions;
    CREATE TABLE test_comparable_predictions AS
    WITH test_base AS (
        SELECT non_price_fingerprint_id, manufacturer, model, manufacturer_model, model_year,
               approximate_vehicle_age, mileage_miles, fuel_type_analysis, transmission_clean,
               engine_size_clean, listed_price_gbp AS actual_listed_price_gbp,
               CASE WHEN approximate_vehicle_age <= 2 THEN '0-2 years'
                    WHEN approximate_vehicle_age <= 5 THEN '3-5 years'
                    WHEN approximate_vehicle_age <= 10 THEN '6-10 years'
                    ELSE '11+ years' END AS age_band
        FROM used_cars_analysis_split WHERE data_split = 'test'
    )
    SELECT t.*,
           COALESCE(ma.comparable_median_price, ml.comparable_median_price, mfa.comparable_median_price,
                    mf.comparable_median_price, ov.comparable_median_price) AS comparable_predicted_price_gbp,
           CASE WHEN ma.comparable_median_price IS NOT NULL THEN '01_model_age'
                WHEN ml.comparable_median_price IS NOT NULL THEN '02_model'
                WHEN mfa.comparable_median_price IS NOT NULL THEN '03_manufacturer_age'
                WHEN mf.comparable_median_price IS NOT NULL THEN '04_manufacturer'
                ELSE '05_overall' END AS comparable_level_used,
           COALESCE(ma.fingerprint_support, ml.fingerprint_support, mfa.fingerprint_support,
                    mf.fingerprint_support, ov.fingerprint_support) AS comparable_fingerprint_support,
           ov.comparable_median_price AS overall_median_predicted_price_gbp
    FROM test_base t
    LEFT JOIN comparable_medians_train ma ON ma.segment_level='01_model_age' AND ma.segment_key = t.manufacturer_model || ' || ' || t.age_band
    LEFT JOIN comparable_medians_train ml ON ml.segment_level='02_model' AND ml.segment_key = t.manufacturer_model
    LEFT JOIN comparable_medians_train mfa ON mfa.segment_level='03_manufacturer_age' AND mfa.segment_key = t.manufacturer || ' || ' || t.age_band
    LEFT JOIN comparable_medians_train mf ON mf.segment_level='04_manufacturer' AND mf.segment_key = t.manufacturer
    CROSS JOIN comparable_medians_train ov
    WHERE ov.segment_level='05_overall' AND ov.segment_key='ALL';
""")

comparable_coverage = pd.read_sql_query("""
    SELECT comparable_level_used, COUNT(*) AS test_records,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM test_comparable_predictions), 2) AS test_record_percentage,
           MIN(comparable_fingerprint_support) AS minimum_fingerprint_support,
           ROUND(AVG(comparable_fingerprint_support),2) AS average_fingerprint_support,
           MAX(comparable_fingerprint_support) AS maximum_fingerprint_support,
           SUM(CASE WHEN comparable_predicted_price_gbp IS NULL THEN 1 ELSE 0 END) AS missing_comparable_predictions
    FROM test_comparable_predictions GROUP BY comparable_level_used ORDER BY comparable_level_used
""", conn)
comparable_coverage

In [ ]:
# one scoring helper - MdAPE is the headline, plus the hit-rate bands.
# reused for every model below so worth pulling out.
def metrics(actual, predicted, name):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    err = predicted - actual
    ape = np.abs(err) / actual * 100
    return {
        "benchmark": name,
        "test_records": len(actual),
        "mae_gbp": np.mean(np.abs(err)),
        "median_absolute_error_gbp": np.median(np.abs(err)),
        "mdape_percentage": np.median(ape),
        "within_10_percent": np.mean(ape <= 10) * 100,
        "within_15_percent": np.mean(ape <= 15) * 100,
        "within_20_percent": np.mean(ape <= 20) * 100,
        "median_signed_percentage_error": np.median(err / actual * 100),
    }

ctr = pd.read_sql_query("""
    SELECT actual_listed_price_gbp, overall_median_predicted_price_gbp, comparable_predicted_price_gbp
    FROM test_comparable_predictions
""", conn)

benchmark_metrics = pd.DataFrame([
    metrics(ctr["actual_listed_price_gbp"], ctr["overall_median_predicted_price_gbp"], "Baseline 0: overall median"),
    metrics(ctr["actual_listed_price_gbp"], ctr["comparable_predicted_price_gbp"], "Baseline 1: hierarchical comparables"),
]).round(2)
benchmark_metrics

## 9. Depreciation & mileage schedules (descriptive)

In [ ]:
age_price_schedule = pd.read_sql_query("""
    WITH ranked AS (
        SELECT approximate_vehicle_age, model_year, listed_price_gbp, non_price_fingerprint_id,
               ROW_NUMBER() OVER (PARTITION BY approximate_vehicle_age ORDER BY listed_price_gbp) AS r,
               COUNT(*) OVER (PARTITION BY approximate_vehicle_age) AS n
        FROM used_cars_analysis
    ),
    stats AS (
        SELECT approximate_vehicle_age, model_year, COUNT(*) AS analysis_records,
               COUNT(DISTINCT non_price_fingerprint_id) AS fingerprint_groups,
               MIN(CASE WHEN r >= 0.25 * n THEN listed_price_gbp END) AS p25_price,
               AVG(CASE WHEN r IN ((n+1)/2, (n+2)/2) THEN listed_price_gbp END) AS median_price,
               ROUND(AVG(listed_price_gbp),2) AS average_price,
               MIN(CASE WHEN r >= 0.75 * n THEN listed_price_gbp END) AS p75_price
        FROM ranked GROUP BY approximate_vehicle_age, model_year
    ),
    ref AS (SELECT median_price AS ref FROM stats WHERE approximate_vehicle_age = 0)
    SELECT a.approximate_vehicle_age, a.model_year, a.analysis_records, a.fingerprint_groups,
           a.p25_price, ROUND(a.median_price,2) AS median_price, a.average_price, a.p75_price,
           ROUND(100.0 * a.median_price / r.ref, 1) AS age_price_index_age_0_equals_100,
           CASE WHEN a.fingerprint_groups >= 25 THEN 'Sufficient support' ELSE 'Low support' END AS support_status
    FROM stats a CROSS JOIN ref r ORDER BY a.approximate_vehicle_age
""", conn)
age_price_schedule

In [ ]:
mileage_price_schedule = pd.read_sql_query("""
    WITH base AS (
        SELECT listed_price_gbp, approximate_vehicle_age, non_price_fingerprint_id,
               CASE WHEN mileage_miles < 5000 THEN '00,000-04,999'
                    WHEN mileage_miles < 10000 THEN '05,000-09,999'
                    WHEN mileage_miles < 20000 THEN '10,000-19,999'
                    WHEN mileage_miles < 30000 THEN '20,000-29,999'
                    WHEN mileage_miles < 40000 THEN '30,000-39,999'
                    WHEN mileage_miles < 60000 THEN '40,000-59,999'
                    WHEN mileage_miles < 80000 THEN '60,000-79,999'
                    WHEN mileage_miles < 100000 THEN '80,000-99,999'
                    WHEN mileage_miles < 150000 THEN '100,000-149,999'
                    ELSE '150,000+' END AS mileage_band,
               CASE WHEN mileage_miles < 5000 THEN 1 WHEN mileage_miles < 10000 THEN 2
                    WHEN mileage_miles < 20000 THEN 3 WHEN mileage_miles < 30000 THEN 4
                    WHEN mileage_miles < 40000 THEN 5 WHEN mileage_miles < 60000 THEN 6
                    WHEN mileage_miles < 80000 THEN 7 WHEN mileage_miles < 100000 THEN 8
                    WHEN mileage_miles < 150000 THEN 9 ELSE 10 END AS mileage_band_order
        FROM used_cars_analysis
    ),
    ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY mileage_band ORDER BY listed_price_gbp) AS r,
               COUNT(*) OVER (PARTITION BY mileage_band) AS n
        FROM base
    ),
    price_stats AS (
        SELECT mileage_band, mileage_band_order, COUNT(*) AS analysis_records,
               COUNT(DISTINCT non_price_fingerprint_id) AS fingerprint_groups,
               MIN(CASE WHEN r >= 0.25 * n THEN listed_price_gbp END) AS p25_price,
               AVG(CASE WHEN r IN ((n+1)/2, (n+2)/2) THEN listed_price_gbp END) AS median_price,
               ROUND(AVG(listed_price_gbp),2) AS average_price,
               MIN(CASE WHEN r >= 0.75 * n THEN listed_price_gbp END) AS p75_price
        FROM ranked GROUP BY mileage_band, mileage_band_order
    ),
    ranked_age AS (
        SELECT mileage_band, approximate_vehicle_age,
               ROW_NUMBER() OVER (PARTITION BY mileage_band ORDER BY approximate_vehicle_age) AS r,
               COUNT(*) OVER (PARTITION BY mileage_band) AS n
        FROM base
    ),
    age_stats AS (
        SELECT mileage_band,
               AVG(CASE WHEN r IN ((n+1)/2, (n+2)/2) THEN approximate_vehicle_age END) AS median_approximate_age
        FROM ranked_age GROUP BY mileage_band
    ),
    ref AS (SELECT median_price AS ref FROM price_stats WHERE mileage_band_order = 1)
    SELECT p.mileage_band, p.analysis_records, p.fingerprint_groups,
           ROUND(a.median_approximate_age,1) AS median_approximate_age,
           p.p25_price, ROUND(p.median_price,2) AS median_price, p.average_price, p.p75_price,
           ROUND(100.0 * p.median_price / r.ref, 1) AS mileage_price_index_lowest_band_equals_100
    FROM price_stats p JOIN age_stats a ON p.mileage_band = a.mileage_band CROSS JOIN ref r
    ORDER BY p.mileage_band_order
""", conn)
mileage_price_schedule

## 10. Regression models

Log price, piecewise age & mileage splines, engine size, fuel, transmission,
plus a model-group fixed effect. Models with <100 fingerprints get pooled
into "<make> | Other models". Clustered SEs on the fingerprint id.

In [ ]:
conn.executescript("""
    DROP TABLE IF EXISTS used_cars_regression_data;
    CREATE TABLE used_cars_regression_data AS
    SELECT s.*,
           CASE WHEN sm.segment_key IS NOT NULL THEN s.manufacturer_model
                ELSE s.manufacturer || ' | Other models' END AS regression_model_group,
           CASE WHEN s.engine_size_clean IS NOT NULL AND s.transmission_clean IS NOT NULL THEN 1 ELSE 0 END AS regression_complete_case_flag,
           s.approximate_vehicle_age AS age_years,
           CASE WHEN s.approximate_vehicle_age > 1 THEN s.approximate_vehicle_age - 1 ELSE 0 END AS age_after_1_year,
           CASE WHEN s.approximate_vehicle_age > 3 THEN s.approximate_vehicle_age - 3 ELSE 0 END AS age_after_3_years,
           CASE WHEN s.approximate_vehicle_age > 5 THEN s.approximate_vehicle_age - 5 ELSE 0 END AS age_after_5_years,
           CASE WHEN s.approximate_vehicle_age > 10 THEN s.approximate_vehicle_age - 10 ELSE 0 END AS age_after_10_years,
           s.mileage_miles / 10000.0 AS mileage_10k,
           CASE WHEN s.mileage_miles > 10000 THEN (s.mileage_miles - 10000) / 10000.0 ELSE 0 END AS mileage_after_10k,
           CASE WHEN s.mileage_miles > 30000 THEN (s.mileage_miles - 30000) / 10000.0 ELSE 0 END AS mileage_after_30k,
           CASE WHEN s.mileage_miles > 60000 THEN (s.mileage_miles - 60000) / 10000.0 ELSE 0 END AS mileage_after_60k,
           CASE WHEN s.mileage_miles > 100000 THEN (s.mileage_miles - 100000) / 10000.0 ELSE 0 END AS mileage_after_100k
    FROM used_cars_analysis_split s
    LEFT JOIN comparable_medians_train sm ON sm.segment_level = '02_model' AND sm.segment_key = s.manufacturer_model;
""")

regression_population_summary = pd.read_sql_query("""
    SELECT data_split, COUNT(*) AS total_records,
           SUM(regression_complete_case_flag) AS complete_case_records,
           COUNT(*) - SUM(regression_complete_case_flag) AS excluded_incomplete_records,
           SUM(CASE WHEN engine_size_clean IS NULL THEN 1 ELSE 0 END) AS unavailable_engine_size,
           SUM(CASE WHEN transmission_clean IS NULL THEN 1 ELSE 0 END) AS unavailable_transmission,
           COUNT(DISTINCT regression_model_group) AS regression_model_groups,
           COUNT(DISTINCT non_price_fingerprint_id) AS fingerprint_groups
    FROM used_cars_regression_data GROUP BY data_split ORDER BY data_split
""", conn)
regression_population_summary

In [ ]:
train_model_data = pd.read_sql_query("""
    SELECT * FROM used_cars_regression_data
    WHERE data_split = 'train' AND regression_complete_case_flag = 1
""", conn)

test_model_data = pd.read_sql_query("""
    SELECT r.*, c.comparable_predicted_price_gbp, c.overall_median_predicted_price_gbp
    FROM used_cars_regression_data r
    JOIN test_comparable_predictions c
      ON r.non_price_fingerprint_id = c.non_price_fingerprint_id
     AND r.listed_price_gbp = c.actual_listed_price_gbp
    WHERE r.data_split = 'test' AND r.regression_complete_case_flag = 1
""", conn)

# A1 - plain log-linear
model_a1 = smf.ols("""
    np.log(listed_price_gbp) ~ age_years + mileage_10k + engine_size_clean
    + C(fuel_type_analysis, Treatment(reference='Petrol'))
    + C(transmission_clean, Treatment(reference='Manual'))
    + C(regression_model_group)
""", data=train_model_data).fit(cov_type="cluster", cov_kwds={"groups": train_model_data["non_price_fingerprint_id"]})

# duan smearing to correct the log->level retransform bias
a1_smear = np.mean(np.exp(model_a1.resid))
test_model_data["model_a1_predicted_price_gbp"] = np.exp(model_a1.predict(test_model_data)) * a1_smear

print("training records:", int(model_a1.nobs))
print("adj R^2:", round(model_a1.rsquared_adj, 4))
print("smearing factor:", round(a1_smear, 4))

pd.DataFrame([
    metrics(test_model_data["listed_price_gbp"], test_model_data["overall_median_predicted_price_gbp"], "Baseline 0: overall median"),
    metrics(test_model_data["listed_price_gbp"], test_model_data["comparable_predicted_price_gbp"], "Baseline 1: hierarchical comparables"),
    metrics(test_model_data["listed_price_gbp"], test_model_data["model_a1_predicted_price_gbp"], "Model A1: simple log-linear regression"),
]).round(2)

In [ ]:
# A2 - piecewise age & mileage. this is the one I end up keeping.
model_a2 = smf.ols("""
    np.log(listed_price_gbp) ~ age_years + age_after_1_year + age_after_3_years
    + age_after_5_years + age_after_10_years + mileage_10k + mileage_after_10k
    + mileage_after_30k + mileage_after_60k + mileage_after_100k + engine_size_clean
    + C(fuel_type_analysis, Treatment(reference='Petrol'))
    + C(transmission_clean, Treatment(reference='Manual'))
    + C(regression_model_group)
""", data=train_model_data).fit(cov_type="cluster", cov_kwds={"groups": train_model_data["non_price_fingerprint_id"]})

a2_smear = np.mean(np.exp(model_a2.resid))
test_model_data["model_a2_predicted_price_gbp"] = np.exp(model_a2.predict(test_model_data)) * a2_smear

print("A2 training records:", int(model_a2.nobs))
print("A2 adj R^2:", round(model_a2.rsquared_adj, 4))
print("A2 smearing factor:", round(a2_smear, 4))

pd.DataFrame([
    metrics(test_model_data["listed_price_gbp"], test_model_data["comparable_predicted_price_gbp"], "Baseline 1: hierarchical comparables"),
    metrics(test_model_data["listed_price_gbp"], test_model_data["model_a1_predicted_price_gbp"], "Model A1: simple log-linear"),
    metrics(test_model_data["listed_price_gbp"], test_model_data["model_a2_predicted_price_gbp"], "Model A2: piecewise nonlinear"),
]).round(2)

### A3: excess-mileage-for-age instead of raw mileage
Age and mileage are collinear, so try re-expressing mileage as the deviation
from the typical mileage at that age (train medians, ages 19+ pooled).

In [ ]:
conn.executescript("""
    DROP TABLE IF EXISTS typical_mileage_by_age_train;
    CREATE TABLE typical_mileage_by_age_train AS
    WITH fp AS (
        SELECT DISTINCT non_price_fingerprint_id, age_years, mileage_miles
        FROM used_cars_regression_data WHERE data_split = 'train' AND regression_complete_case_flag = 1
    ),
    ranked AS (
        SELECT age_years, mileage_miles,
               ROW_NUMBER() OVER (PARTITION BY age_years ORDER BY mileage_miles) AS r,
               COUNT(*) OVER (PARTITION BY age_years) AS n
        FROM fp
    )
    SELECT age_years, MAX(n) AS fingerprint_support,
           ROUND(AVG(CASE WHEN r IN ((n+1)/2, (n+2)/2) THEN mileage_miles END), 2) AS typical_mileage_miles
    FROM ranked GROUP BY age_years;
""")

typical_mileage_by_age = pd.read_sql_query("""
    SELECT age_years, fingerprint_support, typical_mileage_miles
    FROM typical_mileage_by_age_train ORDER BY age_years
""", conn)
typical_mileage_by_age

In [ ]:
conn.executescript("""
    DROP TABLE IF EXISTS typical_mileage_reference_train;
    DROP TABLE IF EXISTS used_cars_regression_excess;

    CREATE TABLE typical_mileage_reference_train AS
    WITH fp AS (
        SELECT DISTINCT non_price_fingerprint_id,
               CASE WHEN age_years >= 19 THEN 19 ELSE age_years END AS age_reference_group,
               mileage_miles
        FROM used_cars_regression_data WHERE data_split = 'train' AND regression_complete_case_flag = 1
    ),
    ranked AS (
        SELECT age_reference_group, mileage_miles,
               ROW_NUMBER() OVER (PARTITION BY age_reference_group ORDER BY mileage_miles) AS r,
               COUNT(*) OVER (PARTITION BY age_reference_group) AS n
        FROM fp
    )
    SELECT age_reference_group, MAX(n) AS fingerprint_support,
           ROUND(AVG(CASE WHEN r IN ((n+1)/2, (n+2)/2) THEN mileage_miles END), 2) AS typical_mileage_miles
    FROM ranked GROUP BY age_reference_group;

    CREATE TABLE used_cars_regression_excess AS
    SELECT r.*, t.typical_mileage_miles,
           (r.mileage_miles - t.typical_mileage_miles) / 10000.0 AS excess_mileage_10k
    FROM used_cars_regression_data r
    JOIN typical_mileage_reference_train t
      ON t.age_reference_group = CASE WHEN r.age_years >= 19 THEN 19 ELSE r.age_years END;
""")

train_excess_data = pd.read_sql_query("""
    SELECT * FROM used_cars_regression_excess
    WHERE data_split = 'train' AND regression_complete_case_flag = 1
""", conn)

test_excess_data = pd.read_sql_query("""
    SELECT e.*, c.comparable_predicted_price_gbp, c.overall_median_predicted_price_gbp
    FROM used_cars_regression_excess e
    JOIN test_comparable_predictions c
      ON e.non_price_fingerprint_id = c.non_price_fingerprint_id
     AND e.listed_price_gbp = c.actual_listed_price_gbp
    WHERE e.data_split = 'test' AND e.regression_complete_case_flag = 1
""", conn)

test_excess_data = test_excess_data.merge(
    test_model_data[["non_price_fingerprint_id", "listed_price_gbp", "model_a1_predicted_price_gbp"]],
    on=["non_price_fingerprint_id", "listed_price_gbp"], how="left", validate="one_to_one"
)

model_a3 = smf.ols("""
    np.log(listed_price_gbp) ~ age_years + excess_mileage_10k + engine_size_clean
    + C(fuel_type_analysis, Treatment(reference='Petrol'))
    + C(transmission_clean, Treatment(reference='Manual'))
    + C(regression_model_group)
""", data=train_excess_data).fit(cov_type="cluster", cov_kwds={"groups": train_excess_data["non_price_fingerprint_id"]})

a3_smear = np.mean(np.exp(model_a3.resid))
test_excess_data["model_a3_predicted_price_gbp"] = np.exp(model_a3.predict(test_excess_data)) * a3_smear

print("A3 adj R^2:", round(model_a3.rsquared_adj, 4))
print("A3 smearing factor:", round(a3_smear, 4))

pd.DataFrame([
    metrics(test_excess_data["listed_price_gbp"], test_excess_data["comparable_predicted_price_gbp"], "Baseline 1: hierarchical comparables"),
    metrics(test_excess_data["listed_price_gbp"], test_excess_data["model_a1_predicted_price_gbp"], "Model A1: direct mileage"),
    metrics(test_excess_data["listed_price_gbp"], test_excess_data["model_a3_predicted_price_gbp"], "Model A3: excess mileage for age"),
]).round(2)

### Where does A3 win/lose vs the comparable baseline?

In [ ]:
sv = test_excess_data.copy()
sv["price_band"] = pd.cut(sv["listed_price_gbp"], [0, 10000, 20000, 30000, 50000, np.inf],
                          labels=["Under £10k", "£10k-£20k", "£20k-£30k", "£30k-£50k", "£50k+"], right=False)
sv["age_band"] = pd.cut(sv["approximate_vehicle_age"], [-1, 2, 5, 10, np.inf],
                        labels=["0-2 years", "3-5 years", "6-10 years", "11+ years"])

rows = []
for dim, col in {"Price band": "price_band", "Manufacturer": "manufacturer", "Age band": "age_band"}.items():
    for seg, frame in sv.groupby(col, observed=True):
        actual = frame["listed_price_gbp"]
        comp_ape = np.abs(frame["comparable_predicted_price_gbp"] - actual) / actual * 100
        mdl_err = frame["model_a3_predicted_price_gbp"] - actual
        mdl_ape = np.abs(mdl_err) / actual * 100
        rows.append({
            "dimension": dim, "segment": str(seg), "test_records": len(frame),
            "comparable_mdape": np.median(comp_ape),
            "model_a3_mdape": np.median(mdl_ape),
            "mdape_improvement_points": np.median(comp_ape) - np.median(mdl_ape),
            "model_a3_mae_gbp": np.mean(np.abs(mdl_err)),
            "model_a3_within_20_percent": np.mean(mdl_ape <= 20) * 100,
            "model_a3_median_signed_error": np.median(mdl_err / actual * 100),
        })
segment_validation = pd.DataFrame(rows).round(2)
segment_validation

In [ ]:
# A1/A2/A3 side by side, by price and age band
msc = test_excess_data.merge(
    test_model_data[["non_price_fingerprint_id", "listed_price_gbp", "model_a2_predicted_price_gbp"]],
    on=["non_price_fingerprint_id", "listed_price_gbp"], how="left", validate="one_to_one"
)
msc["price_band"] = pd.cut(msc["listed_price_gbp"], [0, 10000, 20000, 30000, 50000, np.inf],
                           labels=["Under £10k", "£10k-£20k", "£20k-£30k", "£30k-£50k", "£50k+"], right=False)
msc["age_band"] = pd.cut(msc["approximate_vehicle_age"], [-1, 2, 5, 10, np.inf],
                         labels=["0-2 years", "3-5 years", "6-10 years", "11+ years"])

pred_cols = {"a1": "model_a1_predicted_price_gbp", "a2": "model_a2_predicted_price_gbp", "a3": "model_a3_predicted_price_gbp"}
rows = []
for dim, col in {"Price band": "price_band", "Age band": "age_band"}.items():
    for seg, frame in msc.groupby(col, observed=True):
        row = {"dimension": dim, "segment": str(seg), "test_records": len(frame)}
        actual = frame["listed_price_gbp"]
        for name, pcol in pred_cols.items():
            err = frame[pcol] - actual
            ape = np.abs(err) / actual * 100
            row[f"{name}_mdape"] = np.median(ape)
            row[f"{name}_within_20"] = np.mean(ape <= 20) * 100
            row[f"{name}_median_bias"] = np.median(err / actual * 100)
        rows.append(row)
segment_model_comparison = pd.DataFrame(rows).round(2)
segment_model_comparison

### B1: raw price instead of log
Sanity check the log transform is actually pulling its weight (and isn't
spitting out negative prices).

In [ ]:
model_b1 = smf.ols("""
    listed_price_gbp ~ age_years + age_after_1_year + age_after_3_years + age_after_5_years
    + age_after_10_years + mileage_10k + mileage_after_10k + mileage_after_30k
    + mileage_after_60k + mileage_after_100k + engine_size_clean
    + C(fuel_type_analysis, Treatment(reference='Petrol'))
    + C(transmission_clean, Treatment(reference='Manual'))
    + C(regression_model_group)
""", data=train_model_data).fit(cov_type="cluster", cov_kwds={"groups": train_model_data["non_price_fingerprint_id"]})

test_model_data["model_b1_predicted_price_gbp"] = model_b1.predict(test_model_data)

print("B1 training records:", int(model_b1.nobs))
print("B1 adj R^2:", round(model_b1.rsquared_adj, 4))
print("negative test predictions:", int((test_model_data["model_b1_predicted_price_gbp"] < 0).sum()))

pd.DataFrame([
    metrics(test_model_data["listed_price_gbp"], test_model_data["comparable_predicted_price_gbp"], "Baseline 1: hierarchical comparables"),
    metrics(test_model_data["listed_price_gbp"], test_model_data["model_a2_predicted_price_gbp"], "Model A2: piecewise log price"),
    metrics(test_model_data["listed_price_gbp"], test_model_data["model_b1_predicted_price_gbp"], "Model B1: piecewise raw price"),
]).round(2)

## 11. A2 diagnostics

In [ ]:
# calibration by predicted-price decile
cal = test_model_data[["listed_price_gbp", "model_a2_predicted_price_gbp"]].copy()
cal["predicted_price_decile"] = pd.qcut(cal["model_a2_predicted_price_gbp"], 10,
    labels=["D1 lowest", "D2", "D3", "D4", "D5", "D6", "D7", "D8", "D9", "D10 highest"])

rows = []
for dec, frame in cal.groupby("predicted_price_decile", observed=True):
    actual = frame["listed_price_gbp"]
    pred = frame["model_a2_predicted_price_gbp"]
    err = pred - actual
    ape = np.abs(err) / actual * 100
    rows.append({
        "predicted_price_decile": str(dec), "test_records": len(frame),
        "median_actual_price": np.median(actual), "median_predicted_price": np.median(pred),
        "predicted_to_actual_median_ratio": np.median(pred) / np.median(actual),
        "mdape_percentage": np.median(ape),
        "median_signed_error_percentage": np.median(err / actual * 100),
        "mae_gbp": np.mean(np.abs(err)),
    })
a2_calibration_by_decile = pd.DataFrame(rows).round(2)
a2_calibration_by_decile

In [ ]:
# collinearity: age vs mileage, and VIFs for direct vs excess mileage
age_mileage_relationship = pd.DataFrame([
    {"comparison": "Age vs direct mileage",
     "pearson_correlation": train_model_data[["age_years", "mileage_10k"]].corr("pearson").iloc[0, 1],
     "spearman_correlation": train_model_data[["age_years", "mileage_10k"]].corr("spearman").iloc[0, 1]},
    {"comparison": "Age vs excess mileage",
     "pearson_correlation": train_excess_data[["age_years", "excess_mileage_10k"]].corr("pearson").iloc[0, 1],
     "spearman_correlation": train_excess_data[["age_years", "excess_mileage_10k"]].corr("spearman").iloc[0, 1]},
]).round(3)
age_mileage_relationship

In [ ]:
def vif_table(data, cols, spec):
    v = sm.add_constant(data[cols].dropna().astype(float))
    return [{"specification": spec, "variable": c, "vif": variance_inflation_factor(v.values, i)}
            for i, c in enumerate(v.columns) if c != "const"]

vif_comparison = pd.DataFrame(
    vif_table(train_model_data, ["age_years", "mileage_10k", "engine_size_clean"], "Direct mileage")
    + vif_table(train_excess_data, ["age_years", "excess_mileage_10k", "engine_size_clean"], "Excess mileage")
).round(2)
vif_comparison

In [ ]:
# turn the engine/fuel/transmission coefficients into % price effects
a2_params = np.asarray(model_a2.params)
a2_ci = np.asarray(model_a2.conf_int())
rows = []
for i, term in enumerate(model_a2.params.index):
    if not (term == "engine_size_clean" or "fuel_type_analysis" in term or "transmission_clean" in term):
        continue
    b, lo, hi = a2_params[i], a2_ci[i, 0], a2_ci[i, 1]
    if term == "engine_size_clean":
        label, ref, scale = "Reported engine size: +0.5 unit", "Same observed attributes", 0.5
    elif "fuel_type_analysis" in term:
        label, ref, scale = f"Fuel type: {term.split('[T.')[1].rstrip(']')}", "Petrol", 1.0
    else:
        label, ref, scale = f"Transmission: {term.split('[T.')[1].rstrip(']')}", "Manual", 1.0
    rows.append({
        "feature_comparison": label, "reference": ref,
        "conditional_price_difference_pct": (np.exp(b * scale) - 1) * 100,
        "confidence_interval_low_pct": (np.exp(lo * scale) - 1) * 100,
        "confidence_interval_high_pct": (np.exp(hi * scale) - 1) * 100,
    })
a2_feature_effects = pd.DataFrame(rows).round(2)
a2_feature_effects

### Adjusted age & mileage curves (everything else held fixed)
Use the A2 covariance to put a clustered 95% CI on each contrast.

In [ ]:
params = model_a2.params
covariance = model_a2.cov_params()

def price_index(term_values):
    contrast = pd.Series(0.0, index=params.index)
    for t, v in term_values.items():
        contrast.loc[t] = v
    logdiff = float(contrast @ params)
    se = np.sqrt(max(float(contrast.to_numpy() @ covariance.to_numpy() @ contrast.to_numpy()), 0))
    return 100 * np.exp(logdiff), 100 * np.exp(logdiff - 1.96 * se), 100 * np.exp(logdiff + 1.96 * se)

# age curve, ref = age 0. drop 19+ (thin support)
age_rows = []
for age in range(0, 19):
    idx, lo, hi = price_index({
        "age_years": age, "age_after_1_year": max(age - 1, 0), "age_after_3_years": max(age - 3, 0),
        "age_after_5_years": max(age - 5, 0), "age_after_10_years": max(age - 10, 0),
    })
    age_rows.append({"approximate_vehicle_age": age, "model_year": 2020 - age,
                     "adjusted_price_index_age_0_equals_100": idx,
                     "confidence_interval_low": lo, "confidence_interval_high": hi})
adjusted_age_curve = pd.DataFrame(age_rows).round(1)

# mileage curve, ref = 0 miles
mileage_rows = []
for m in [0, 5000, 10000, 20000, 30000, 40000, 60000, 80000, 100000, 150000]:
    m10 = m / 10000
    idx, lo, hi = price_index({
        "mileage_10k": m10, "mileage_after_10k": max(m10 - 1, 0), "mileage_after_30k": max(m10 - 3, 0),
        "mileage_after_60k": max(m10 - 6, 0), "mileage_after_100k": max(m10 - 10, 0),
    })
    mileage_rows.append({"mileage_miles": m, "adjusted_price_index_0_miles_equals_100": idx,
                         "confidence_interval_low": lo, "confidence_interval_high": hi})
adjusted_mileage_curve = pd.DataFrame(mileage_rows).round(1)

print("Adjusted age relationship")
display(adjusted_age_curve)
print("Adjusted mileage relationship")
display(adjusted_mileage_curve)

### Make- and model-level price indices
Fingerprint-weighted portfolio average = 100, with CIs from the model-group
dummies' covariance.

In [ ]:
model_frame = model_a2.model.data.frame.copy()

model_support = (model_frame.groupby("regression_model_group")["non_price_fingerprint_id"]
                 .nunique().rename("fingerprint_support").reset_index())
model_support["manufacturer"] = model_support["regression_model_group"].str.split(" | ", n=1, regex=False).str[0]

pidx = {name: pos for pos, name in enumerate(params.index)}

# design vector for a model group (omitted reference gets all zeros)
def group_vector(group):
    v = np.zeros(len(params))
    name = f"C(regression_model_group)[T.{group}]"
    if name in pidx:
        v[pidx[name]] = 1.0
    return v

model_support["design_vector"] = model_support["regression_model_group"].apply(group_vector)

total_support = model_support["fingerprint_support"].sum()
portfolio_vector = sum(r["design_vector"] * r["fingerprint_support"] for _, r in model_support.iterrows()) / total_support

def contrast_index(contrast):
    est = float(contrast @ params.to_numpy())
    se = np.sqrt(max(float(contrast @ covariance.to_numpy() @ contrast), 0))
    return 100 * np.exp(est), 100 * np.exp(est - 1.96 * se), 100 * np.exp(est + 1.96 * se)

# by manufacturer
rows = []
for mfr, grp in model_support.groupby("manufacturer"):
    supp = grp["fingerprint_support"].sum()
    vec = sum(r["design_vector"] * r["fingerprint_support"] for _, r in grp.iterrows()) / supp
    idx, lo, hi = contrast_index(vec - portfolio_vector)
    rows.append({"manufacturer": mfr, "fingerprint_support": supp, "named_and_pooled_model_groups": len(grp),
                 "conditional_price_index_portfolio_equals_100": idx,
                 "confidence_interval_low": lo, "confidence_interval_high": hi})
manufacturer_price_indices = pd.DataFrame(rows).sort_values(
    "conditional_price_index_portfolio_equals_100", ascending=False).round(1)

# named models with >=100 fingerprints (drop the pooled "Other models")
rows = []
for _, r in model_support.iterrows():
    grp = r["regression_model_group"]
    if r["fingerprint_support"] < 100 or grp.endswith("Other models"):
        continue
    idx, lo, hi = contrast_index(r["design_vector"] - portfolio_vector)
    rows.append({"manufacturer_model": grp, "fingerprint_support": r["fingerprint_support"],
                 "conditional_price_index_portfolio_equals_100": idx,
                 "confidence_interval_low": lo, "confidence_interval_high": hi})
model_price_indices = pd.DataFrame(rows).round(1)
highest_model_indices = model_price_indices.sort_values("conditional_price_index_portfolio_equals_100", ascending=False).head(10)
lowest_model_indices = model_price_indices.sort_values("conditional_price_index_portfolio_equals_100").head(10)

print("Manufacturer-level conditional price indices")
display(manufacturer_price_indices)
print("Ten highest supported named-model indices")
display(highest_model_indices)
print("Ten lowest supported named-model indices")
display(lowest_model_indices)

### Held-out price positioning
A2 gives an expected listed price; the gap vs actual flags cars listed
unusually high or low. Tails at 2.5% each side.

In [ ]:
diagnostic_test = pd.read_sql_query("""
    SELECT * FROM used_cars_regression_data
    WHERE data_split = 'test' AND engine_size_clean IS NOT NULL AND transmission_clean IS NOT NULL
""", conn).copy()

# rebuild the A2 piecewise cols on this frame
diagnostic_test["age_after_1_year"] = np.maximum(diagnostic_test["age_years"] - 1, 0)
diagnostic_test["age_after_3_years"] = np.maximum(diagnostic_test["age_years"] - 3, 0)
diagnostic_test["age_after_5_years"] = np.maximum(diagnostic_test["age_years"] - 5, 0)
diagnostic_test["age_after_10_years"] = np.maximum(diagnostic_test["age_years"] - 10, 0)
diagnostic_test["mileage_10k"] = diagnostic_test["mileage_miles"] / 10000
diagnostic_test["mileage_after_10k"] = np.maximum(diagnostic_test["mileage_10k"] - 1, 0)
diagnostic_test["mileage_after_30k"] = np.maximum(diagnostic_test["mileage_10k"] - 3, 0)
diagnostic_test["mileage_after_60k"] = np.maximum(diagnostic_test["mileage_10k"] - 6, 0)
diagnostic_test["mileage_after_100k"] = np.maximum(diagnostic_test["mileage_10k"] - 10, 0)

qa_smear = np.mean(np.exp(model_a2.resid))
diagnostic_test["expected_listed_price_gbp"] = (np.exp(model_a2.predict(diagnostic_test)) * qa_smear).round(2)
diagnostic_test["price_position_gap_gbp"] = diagnostic_test["listed_price_gbp"] - diagnostic_test["expected_listed_price_gbp"]
diagnostic_test["price_position_gap_pct"] = 100 * diagnostic_test["price_position_gap_gbp"] / diagnostic_test["expected_listed_price_gbp"]

pctls = [0.0, 0.01, 0.025, 0.05, 0.25, 0.5, 0.75, 0.95, 0.975, 0.99, 1.0]
position_distribution = (diagnostic_test["price_position_gap_pct"].quantile(pctls)
                         .rename_axis("percentile").reset_index(name="price_position_gap_pct"))
position_distribution["percentile"] *= 100
position_distribution = position_distribution.round(2)

lo_thr = diagnostic_test["price_position_gap_pct"].quantile(0.025)
hi_thr = diagnostic_test["price_position_gap_pct"].quantile(0.975)
diagnostic_test["positioning_diagnostic"] = np.select(
    [diagnostic_test["price_position_gap_pct"] <= lo_thr, diagnostic_test["price_position_gap_pct"] >= hi_thr],
    ["Lower 2.5% versus model benchmark", "Upper 2.5% versus model benchmark"],
    default="Central 95%")

out_cols = ["manufacturer", "model", "model_year", "approximate_vehicle_age", "mileage_miles",
            "listed_price_gbp", "expected_listed_price_gbp", "price_position_gap_gbp",
            "price_position_gap_pct", "positioning_diagnostic"]
lowest_positioned_listings = diagnostic_test.nsmallest(10, "price_position_gap_pct")[out_cols].round(2)
highest_positioned_listings = diagnostic_test.nlargest(10, "price_position_gap_pct")[out_cols].round(2)

diagnostic_summary = pd.DataFrame({
    "test_records": [len(diagnostic_test)],
    "lower_2_5_percent_threshold": [lo_thr], "upper_2_5_percent_threshold": [hi_thr],
    "lower_tail_records": [(diagnostic_test["positioning_diagnostic"] == "Lower 2.5% versus model benchmark").sum()],
    "upper_tail_records": [(diagnostic_test["positioning_diagnostic"] == "Upper 2.5% versus model benchmark").sum()],
}).round(2)

print("Held-out price-position distribution")
display(position_distribution)
print("Diagnostic thresholds and record counts")
display(diagnostic_summary)
print("Listings furthest below the model benchmark")
display(lowest_positioned_listings)
print("Listings furthest above the model benchmark")
display(highest_positioned_listings)

## 12. Strengthened comparable (adds a mileage band)
Give the comparable baseline the fairest shot possible - model x age x
mileage at the top of the hierarchy. It still loses to A2, which is the point.

In [ ]:
strong_comparable_train = pd.read_sql_query("""
    SELECT manufacturer, model, approximate_vehicle_age, mileage_miles, listed_price_gbp, non_price_fingerprint_id
    FROM used_cars_regression_data WHERE data_split = 'train'
""", conn).copy()
strong_comparable_test = diagnostic_test.copy()

# same segment bins get applied to train, test and later the full portfolio
def add_segments(d):
    d = d.copy()
    d["manufacturer_model"] = d["manufacturer"].astype(str) + " | " + d["model"].astype(str)
    d["age_band"] = pd.cut(d["approximate_vehicle_age"], [-np.inf, 2, 5, 10, np.inf],
                           labels=["0-2 years", "3-5 years", "6-10 years", "11+ years"])
    d["mileage_band"] = pd.cut(d["mileage_miles"],
        [-np.inf, 4999, 9999, 19999, 29999, 39999, 59999, 79999, 99999, 149999, np.inf],
        labels=["00,000-04,999", "05,000-09,999", "10,000-19,999", "20,000-29,999", "30,000-39,999",
                "40,000-59,999", "60,000-79,999", "80,000-99,999", "100,000-149,999", "150,000+"])
    return d

strong_comparable_train = add_segments(strong_comparable_train)
strong_comparable_test = add_segments(strong_comparable_test)

comparable_levels = [
    ("01_model_age_mileage", ["manufacturer_model", "age_band", "mileage_band"]),
    ("02_model_age", ["manufacturer_model", "age_band"]),
    ("03_model", ["manufacturer_model"]),
    ("04_manufacturer_age", ["manufacturer", "age_band"]),
    ("05_manufacturer", ["manufacturer"]),
]

# build the train-only lookup per level (>=25 fingerprints)
comparable_lookups = {}
for lvl, keys in comparable_levels:
    t = (strong_comparable_train.groupby(keys, observed=True, dropna=False)
         .agg(comparable_median_price=("listed_price_gbp", "median"),
              fingerprint_support=("non_price_fingerprint_id", "nunique"))
         .reset_index())
    comparable_lookups[lvl] = t[t["fingerprint_support"] >= 25].copy()

overall_train_median = strong_comparable_train["listed_price_gbp"].median()
overall_fingerprint_support = strong_comparable_train["non_price_fingerprint_id"].nunique()

# walk the test rows down the hierarchy
strong_comparable_test["strong_comparable_prediction"] = np.nan
strong_comparable_test["strong_comparable_level"] = None
strong_comparable_test["strong_comparable_support"] = np.nan
for lvl, keys in comparable_levels:
    lk = comparable_lookups[lvl]
    li = pd.MultiIndex.from_frame(lk[keys])
    price_lk = pd.Series(lk["comparable_median_price"].to_numpy(), index=li)
    supp_lk = pd.Series(lk["fingerprint_support"].to_numpy(), index=li)
    ti = pd.MultiIndex.from_frame(strong_comparable_test[keys])
    cand_price = price_lk.reindex(ti).to_numpy()
    cand_supp = supp_lk.reindex(ti).to_numpy()
    use = strong_comparable_test["strong_comparable_prediction"].isna() & pd.notna(cand_price)
    strong_comparable_test.loc[use, "strong_comparable_prediction"] = cand_price[use]
    strong_comparable_test.loc[use, "strong_comparable_support"] = cand_supp[use]
    strong_comparable_test.loc[use, "strong_comparable_level"] = lvl
need = strong_comparable_test["strong_comparable_prediction"].isna()
strong_comparable_test.loc[need, "strong_comparable_prediction"] = overall_train_median
strong_comparable_test.loc[need, "strong_comparable_support"] = overall_fingerprint_support
strong_comparable_test.loc[need, "strong_comparable_level"] = "06_overall"

strong_comparable_coverage = (strong_comparable_test.groupby("strong_comparable_level", dropna=False)
    .agg(test_records=("non_price_fingerprint_id", "size"),
         fingerprint_groups=("non_price_fingerprint_id", "nunique"),
         minimum_fingerprint_support=("strong_comparable_support", "min"),
         average_fingerprint_support=("strong_comparable_support", "mean"),
         maximum_fingerprint_support=("strong_comparable_support", "max"))
    .reset_index())
strong_comparable_coverage["test_record_percentage"] = 100 * strong_comparable_coverage["test_records"] / len(strong_comparable_test)
strong_comparable_coverage = strong_comparable_coverage.sort_values("strong_comparable_level").round(2)

strong_baseline_metrics = pd.DataFrame([
    metrics(strong_comparable_test["listed_price_gbp"], strong_comparable_test["strong_comparable_prediction"], "Strengthened hierarchical comparables"),
    metrics(strong_comparable_test["listed_price_gbp"], strong_comparable_test["expected_listed_price_gbp"], "Model A2: piecewise log price"),
]).round(2)

print("Strengthened comparable coverage")
display(strong_comparable_coverage)
print("Strengthened baseline versus Model A2")
display(strong_baseline_metrics)

## 13. Joint age x mileage schedule
The headline output: what A2 says a car is worth at standard age / mileage
combos, with the training support behind each cell.

In [ ]:
selected_ages = [0, 1, 3, 5, 7, 10]
selected_mileage_points = {0: "00,000-04,999", 10000: "10,000-19,999", 30000: "30,000-39,999",
                           60000: "60,000-79,999", 100000: "100,000-149,999"}

joint_support = (strong_comparable_train[
        strong_comparable_train["approximate_vehicle_age"].isin(selected_ages)
        & strong_comparable_train["mileage_band"].astype(str).isin(selected_mileage_points.values())]
    .groupby(["approximate_vehicle_age", "mileage_band"], observed=True)["non_price_fingerprint_id"]
    .nunique().rename("training_fingerprint_support"))

rows = []
for age in selected_ages:
    for mileage, band in selected_mileage_points.items():
        m10 = mileage / 10000
        idx, lo, hi = price_index({
            "age_years": age, "age_after_1_year": max(age - 1, 0), "age_after_3_years": max(age - 3, 0),
            "age_after_5_years": max(age - 5, 0), "age_after_10_years": max(age - 10, 0),
            "mileage_10k": m10, "mileage_after_10k": max(m10 - 1, 0), "mileage_after_30k": max(m10 - 3, 0),
            "mileage_after_60k": max(m10 - 6, 0), "mileage_after_100k": max(m10 - 10, 0),
        })
        supp = joint_support.get((age, band), 0)
        status = "Strong support" if supp >= 100 else "Adequate support" if supp >= 25 else "Low support / directional only"
        rows.append({"approximate_vehicle_age": age, "model_year": 2020 - age,
                     "mileage_reference_miles": mileage, "mileage_support_band": band,
                     "joint_adjusted_price_index": idx, "confidence_interval_low": lo,
                     "confidence_interval_high": hi, "training_fingerprint_support": supp,
                     "support_status": status})
joint_age_mileage_schedule = pd.DataFrame(rows).round(
    {"joint_adjusted_price_index": 1, "confidence_interval_low": 1, "confidence_interval_high": 1})

joint_price_index_matrix = joint_age_mileage_schedule.pivot(
    index="approximate_vehicle_age", columns="mileage_reference_miles", values="joint_adjusted_price_index").round(1)
joint_price_index_matrix.columns = [f"{int(m):,} miles" for m in joint_price_index_matrix.columns]

joint_support_matrix = joint_age_mileage_schedule.pivot(
    index="approximate_vehicle_age", columns="mileage_reference_miles", values="training_fingerprint_support")
joint_support_matrix.columns = [f"{int(m):,} miles" for m in joint_support_matrix.columns]

print("Joint adjusted price index: age 0 and mileage 0 = 100")
display(joint_price_index_matrix)
print("Training fingerprint support around each combination")
display(joint_support_matrix)
print("Detailed joint schedule with confidence and support")
display(joint_age_mileage_schedule)

## 14. Reproducibility checks
Recompute the population counts, leakage, A2 fit stats and the two headline
metric sets, and stop hard if anything drifts.

In [ ]:
qa_results = []
def check(name, condition, observed):
    qa_results.append({"check": name, "status": "PASS" if bool(condition) else "FAIL", "observed": str(observed)})

population_audit = pd.read_sql_query("""
    SELECT data_split, COUNT(*) AS total_records,
           COUNT(DISTINCT non_price_fingerprint_id) AS total_fingerprints,
           SUM(CASE WHEN engine_size_clean IS NOT NULL AND transmission_clean IS NOT NULL THEN 1 ELSE 0 END) AS complete_case_records,
           COUNT(DISTINCT CASE WHEN engine_size_clean IS NOT NULL AND transmission_clean IS NOT NULL THEN non_price_fingerprint_id END) AS complete_case_fingerprints
    FROM used_cars_regression_data GROUP BY data_split ORDER BY data_split
""", conn)
pop = population_audit.set_index("data_split").to_dict("index")

expected_population = {
    "train": {"total_records": 78173, "total_fingerprints": 76755, "complete_case_records": 77965, "complete_case_fingerprints": 76555},
    "test": {"total_records": 19500, "total_fingerprints": 19190, "complete_case_records": 19433, "complete_case_fingerprints": 19125},
}
for split, exp in expected_population.items():
    for field, val in exp.items():
        check(f"{split}: {field}", int(pop[split][field]) == val, int(pop[split][field]))

overlap = pd.read_sql_query("""
    SELECT COUNT(*) AS overlapping_fingerprints FROM (
        SELECT non_price_fingerprint_id FROM used_cars_regression_data
        GROUP BY non_price_fingerprint_id HAVING COUNT(DISTINCT data_split) > 1
    )
""", conn).iloc[0]["overlapping_fingerprints"]
check("Zero fingerprints shared across train and test", int(overlap) == 0, int(overlap))

required_a2_terms = {"age_years", "age_after_1_year", "age_after_3_years", "age_after_5_years", "age_after_10_years",
                     "mileage_10k", "mileage_after_10k", "mileage_after_30k", "mileage_after_60k", "mileage_after_100k", "engine_size_clean"}
check("All required A2 terms present", required_a2_terms <= set(model_a2.params.index),
      sorted(required_a2_terms - set(model_a2.params.index)))
check("A2 training records", int(model_a2.nobs) == 77965, int(model_a2.nobs))
check("A2 complete-case training fingerprints", model_frame["non_price_fingerprint_id"].nunique() == 76555,
      model_frame["non_price_fingerprint_id"].nunique())
check("A2 regression model groups", model_frame["regression_model_group"].nunique() == 147,
      model_frame["regression_model_group"].nunique())
check("A2 adjusted R-squared", np.isclose(model_a2.rsquared_adj, 0.9441, atol=1e-4), round(model_a2.rsquared_adj, 4))
check("A2 smearing factor", np.isclose(qa_smear, 1.0081, atol=1e-4), round(qa_smear, 4))
electric_terms = [t for t in model_a2.params.index if "Electric" in t]
check("No unsupported standalone Electric coefficient", len(electric_terms) == 0, electric_terms)

check("Diagnostic test records", len(diagnostic_test) == 19433, len(diagnostic_test))
check("Diagnostic test fingerprints", diagnostic_test["non_price_fingerprint_id"].nunique() == 19125,
      diagnostic_test["non_price_fingerprint_id"].nunique())
check("No missing A2 predictions", diagnostic_test["expected_listed_price_gbp"].notna().all(),
      diagnostic_test["expected_listed_price_gbp"].isna().sum())
check("All A2 predictions positive", (diagnostic_test["expected_listed_price_gbp"] > 0).all(),
      diagnostic_test["expected_listed_price_gbp"].min())
check("No missing strengthened-comparable predictions", strong_comparable_test["strong_comparable_prediction"].notna().all(),
      strong_comparable_test["strong_comparable_prediction"].isna().sum())

# recompute the two headline metric sets and check against what I recorded
def qa_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    ape = 100 * np.abs(predicted - actual) / actual
    return {"mae": np.abs(predicted - actual).mean(), "median_ae": np.median(np.abs(predicted - actual)),
            "mdape": np.median(ape), "within_10": 100 * np.mean(ape <= 10),
            "within_15": 100 * np.mean(ape <= 15), "within_20": 100 * np.mean(ape <= 20),
            "median_bias": np.median(100 * (predicted - actual) / actual)}

a2_qa = qa_metrics(diagnostic_test["listed_price_gbp"], diagnostic_test["expected_listed_price_gbp"])
comp_qa = qa_metrics(strong_comparable_test["listed_price_gbp"], strong_comparable_test["strong_comparable_prediction"])

expected_a2 = {"mae": 1613.11, "median_ae": 1057.47, "mdape": 7.77, "within_10": 61.28, "within_15": 80.04, "within_20": 90.35, "median_bias": 1.03}
expected_comp = {"mae": 2355.01, "median_ae": 1335.00, "mdape": 9.61, "within_10": 51.38, "within_15": 67.65, "within_20": 78.60, "median_bias": 0.07}
for k, v in expected_a2.items():
    check(f"A2 metric: {k}", np.isclose(a2_qa[k], v, atol=0.02), round(a2_qa[k], 2))
for k, v in expected_comp.items():
    check(f"Comparable metric: {k}", np.isclose(comp_qa[k], v, atol=0.02), round(comp_qa[k], 2))

expected_level_counts = {"01_model_age_mileage": 16517, "02_model_age": 2486, "03_model": 330, "04_manufacturer_age": 99, "05_manufacturer": 1}
observed_level_counts = strong_comparable_test["strong_comparable_level"].value_counts().to_dict()
for lvl, cnt in expected_level_counts.items():
    check(f"Comparable coverage: {lvl}", observed_level_counts.get(lvl, 0) == cnt, observed_level_counts.get(lvl, 0))
check("Comparable hierarchy covers all held-out records", sum(observed_level_counts.values()) == 19433, sum(observed_level_counts.values()))

step_4_qa = pd.DataFrame(qa_results)
display(step_4_qa)
failed = step_4_qa[step_4_qa["status"] == "FAIL"]
if len(failed) == 0:
    print("ALL CHECKS PASSED")
else:
    raise AssertionError(f"{len(failed)} checks failed.")

## 15. Empirical price ranges
Split the held-out set in half (by fingerprint hash), learn multiplicative
limits on the actual/expected ratio on one half, check coverage on the other.

In [ ]:
interval_data = diagnostic_test.copy()
fp_hash = pd.util.hash_pandas_object(interval_data["non_price_fingerprint_id"].astype(str), index=False).astype("uint64")
interval_data["interval_sample"] = np.where(fp_hash % 2 == 0, "calibration", "validation")

calibration_data = interval_data[interval_data["interval_sample"] == "calibration"].copy()
interval_validation_data = interval_data[interval_data["interval_sample"] == "validation"].copy()
interval_fingerprint_overlap = len(set(calibration_data["non_price_fingerprint_id"]) & set(interval_validation_data["non_price_fingerprint_id"]))

calibration_data["actual_to_expected_ratio"] = calibration_data["listed_price_gbp"] / calibration_data["expected_listed_price_gbp"]

interval_limits = {
    "90% empirical range": {"lower_multiplier": calibration_data["actual_to_expected_ratio"].quantile(0.05),
                            "upper_multiplier": calibration_data["actual_to_expected_ratio"].quantile(0.95)},
    "95% empirical range": {"lower_multiplier": calibration_data["actual_to_expected_ratio"].quantile(0.025),
                            "upper_multiplier": calibration_data["actual_to_expected_ratio"].quantile(0.975)},
}

for name, lim in interval_limits.items():
    s = "90" if name.startswith("90") else "95"
    interval_validation_data[f"lower_{s}_price"] = interval_validation_data["expected_listed_price_gbp"] * lim["lower_multiplier"]
    interval_validation_data[f"upper_{s}_price"] = interval_validation_data["expected_listed_price_gbp"] * lim["upper_multiplier"]
    interval_validation_data[f"inside_{s}_range"] = interval_validation_data["listed_price_gbp"].between(
        interval_validation_data[f"lower_{s}_price"], interval_validation_data[f"upper_{s}_price"], inclusive="both")

interval_summary = []
for name, lim in interval_limits.items():
    s = "90" if name.startswith("90") else "95"
    interval_summary.append({
        "empirical_range": name,
        "calibration_records": len(calibration_data),
        "calibration_fingerprints": calibration_data["non_price_fingerprint_id"].nunique(),
        "validation_records": len(interval_validation_data),
        "validation_fingerprints": interval_validation_data["non_price_fingerprint_id"].nunique(),
        "fingerprints_in_both_samples": interval_fingerprint_overlap,
        "lower_price_multiplier": lim["lower_multiplier"], "upper_price_multiplier": lim["upper_multiplier"],
        "validation_coverage_percentage": 100 * interval_validation_data[f"inside_{s}_range"].mean(),
        "example_lower_price_for_15000": 15000 * lim["lower_multiplier"],
        "example_upper_price_for_15000": 15000 * lim["upper_multiplier"],
    })
interval_coverage_summary = pd.DataFrame(interval_summary).round(2)

interval_validation_data["age_band"] = pd.cut(interval_validation_data["approximate_vehicle_age"],
    [-np.inf, 2, 5, 10, np.inf], labels=["0-2 years", "3-5 years", "6-10 years", "11+ years"])
interval_validation_data["mileage_band"] = pd.cut(interval_validation_data["mileage_miles"],
    [-np.inf, 9999, 29999, 59999, 99999, np.inf], labels=["Under 10k", "10k-29,999", "30k-59,999", "60k-99,999", "100k+"])
interval_validation_data["price_band"] = pd.cut(interval_validation_data["listed_price_gbp"],
    [-np.inf, 9999, 19999, 29999, 49999, np.inf], labels=["Under £10k", "£10k-£20k", "£20k-£30k", "£30k-£50k", "£50k+"])

segment_tables = []
for dim_name, field in [("Manufacturer", "manufacturer"), ("Age band", "age_band"), ("Mileage band", "mileage_band"), ("Price band", "price_band")]:
    res = (interval_validation_data.groupby(field, observed=True, dropna=False)
           .agg(validation_records=("listed_price_gbp", "size"),
                coverage_90_percentage=("inside_90_range", lambda v: 100 * v.mean()),
                coverage_95_percentage=("inside_95_range", lambda v: 100 * v.mean()))
           .reset_index().rename(columns={field: "segment"}))
    res.insert(0, "dimension", dim_name)
    segment_tables.append(res)
interval_segment_coverage = pd.concat(segment_tables, ignore_index=True).round(2)

print("Empirical price-range calibration and validation")
display(interval_coverage_summary)
print("Empirical price-range coverage by segment")
display(interval_segment_coverage)

## 16. Guidance tiers
Route each listing to standard / cautious / manual based on where A2 is
trustworthy. Anything predicted at £50k+ always goes to manual review.

In [ ]:
tier_population = pd.read_sql_query("SELECT * FROM used_cars_regression_data", conn).copy()
assert len(tier_population) == 97673
tier_population = add_segments(tier_population)

# A2 benchmark where the inputs are complete
tier_population["age_after_1_year"] = np.maximum(tier_population["age_years"] - 1, 0)
tier_population["age_after_3_years"] = np.maximum(tier_population["age_years"] - 3, 0)
tier_population["age_after_5_years"] = np.maximum(tier_population["age_years"] - 5, 0)
tier_population["age_after_10_years"] = np.maximum(tier_population["age_years"] - 10, 0)
tier_population["mileage_10k"] = tier_population["mileage_miles"] / 10000
tier_population["mileage_after_10k"] = np.maximum(tier_population["mileage_10k"] - 1, 0)
tier_population["mileage_after_30k"] = np.maximum(tier_population["mileage_10k"] - 3, 0)
tier_population["mileage_after_60k"] = np.maximum(tier_population["mileage_10k"] - 6, 0)
tier_population["mileage_after_100k"] = np.maximum(tier_population["mileage_10k"] - 10, 0)

complete_for_a2 = tier_population["engine_size_clean"].notna() & tier_population["transmission_clean"].notna()
tier_population["expected_listed_price_gbp"] = np.nan
tier_population.loc[complete_for_a2, "expected_listed_price_gbp"] = (
    np.exp(model_a2.predict(tier_population.loc[complete_for_a2])) * qa_smear)

# strengthened comparable across the whole portfolio (same walk as above)
tier_population["comparable_level"] = None
tier_population["comparable_support"] = np.nan
tier_population["comparable_median_price"] = np.nan
for lvl, keys in comparable_levels:
    lk = comparable_lookups[lvl]
    li = pd.MultiIndex.from_frame(lk[keys])
    price_lk = pd.Series(lk["comparable_median_price"].to_numpy(), index=li)
    supp_lk = pd.Series(lk["fingerprint_support"].to_numpy(), index=li)
    pi = pd.MultiIndex.from_frame(tier_population[keys])
    cand_price = price_lk.reindex(pi).to_numpy()
    cand_supp = supp_lk.reindex(pi).to_numpy()
    use = tier_population["comparable_median_price"].isna() & pd.notna(cand_price)
    tier_population.loc[use, "comparable_median_price"] = cand_price[use]
    tier_population.loc[use, "comparable_support"] = cand_supp[use]
    tier_population.loc[use, "comparable_level"] = lvl
need = tier_population["comparable_median_price"].isna()
tier_population.loc[need, "comparable_median_price"] = overall_train_median
tier_population.loc[need, "comparable_support"] = overall_fingerprint_support
tier_population.loc[need, "comparable_level"] = "06_overall"

pooled_model = tier_population["regression_model_group"].fillna("").str.endswith("Other models")
tier_3 = (
    (tier_population["approximate_vehicle_age"] >= 11)
    | (tier_population["mileage_miles"] >= 100000)
    | (tier_population["fuel_type_analysis"] == "Other / unclear")
    | tier_population["engine_size_clean"].isna()
    | tier_population["transmission_clean"].isna()
    | pooled_model
    | tier_population["expected_listed_price_gbp"].isna()
)
tier_2 = ~tier_3 & (
    tier_population["approximate_vehicle_age"].between(6, 10)
    | tier_population["mileage_miles"].between(60000, 99999)
    | (tier_population["comparable_level"] != "01_model_age_mileage")
)
tier_population["guidance_tier"] = np.select(
    [tier_3, tier_2], ["Tier 3 — Manual appraisal", "Tier 2 — Cautious guidance"], default="Tier 1 — Standard guidance")

total_listed = tier_population["listed_price_gbp"].sum()
tier_summary = (tier_population.groupby("guidance_tier")
    .agg(listings=("listed_price_gbp", "size"),
         aggregate_listed_price_value=("listed_price_gbp", "sum"),
         median_listed_price=("listed_price_gbp", "median"),
         median_expected_price=("expected_listed_price_gbp", "median"))
    .reset_index())
tier_summary["listing_share_percentage"] = 100 * tier_summary["listings"] / len(tier_population)
tier_summary["aggregate_listed_price_share_percentage"] = 100 * tier_summary["aggregate_listed_price_value"] / total_listed
tier_summary = tier_summary[["guidance_tier", "listings", "listing_share_percentage", "aggregate_listed_price_value",
    "aggregate_listed_price_share_percentage", "median_listed_price", "median_expected_price"]].round(2)

# validate the predicted-price triggers on the fingerprint-separated validation half
ppv = interval_validation_data.copy()
ppv["predicted_price_band"] = pd.cut(ppv["expected_listed_price_gbp"], [-np.inf, 29999.99, 49999.99, np.inf],
    labels=["Predicted below £30k", "Predicted £30k-£49,999", "Predicted £50k+"])
ppv["absolute_percentage_error"] = 100 * (ppv["expected_listed_price_gbp"] - ppv["listed_price_gbp"]).abs() / ppv["listed_price_gbp"]
ppv["signed_percentage_error"] = 100 * (ppv["expected_listed_price_gbp"] - ppv["listed_price_gbp"]) / ppv["listed_price_gbp"]
predicted_price_band_validation = (ppv.groupby("predicted_price_band", observed=True)
    .agg(validation_records=("listed_price_gbp", "size"),
         median_actual_price=("listed_price_gbp", "median"),
         median_predicted_price=("expected_listed_price_gbp", "median"),
         mdape_percentage=("absolute_percentage_error", "median"),
         within_20_percent=("absolute_percentage_error", lambda v: 100 * (v <= 20).mean()),
         median_signed_error=("signed_percentage_error", "median"),
         empirical_90_range_coverage=("inside_90_range", lambda v: 100 * v.mean()),
         empirical_95_range_coverage=("inside_95_range", lambda v: 100 * v.mean()))
    .reset_index().round(2))

print("Guidance tier sizing on the locked population")
display(tier_summary)
print("Validation by predicted-price band")
display(predicted_price_band_validation)

In [ ]:
# final rule: bump every predicted-£50k+ car up to manual review
tier_population["guidance_tier_final"] = tier_population["guidance_tier"]
predicted_50k_plus = tier_population["expected_listed_price_gbp"] >= 50000
tier_population.loc[predicted_50k_plus, "guidance_tier_final"] = "Tier 3 — Manual appraisal"

final_tier_summary = (tier_population.groupby("guidance_tier_final")
    .agg(listings=("listed_price_gbp", "size"),
         aggregate_listed_price_value=("listed_price_gbp", "sum"),
         median_listed_price=("listed_price_gbp", "median"),
         median_expected_price=("expected_listed_price_gbp", "median"),
         predicted_50k_plus_records=("expected_listed_price_gbp", lambda v: (v >= 50000).sum()))
    .reset_index())
final_tier_summary["listing_share_percentage"] = 100 * final_tier_summary["listings"] / len(tier_population)
final_tier_summary["aggregate_listed_price_share_percentage"] = 100 * final_tier_summary["aggregate_listed_price_value"] / total_listed
final_tier_summary = final_tier_summary[["guidance_tier_final", "listings", "listing_share_percentage",
    "aggregate_listed_price_value", "aggregate_listed_price_share_percentage", "median_listed_price",
    "median_expected_price", "predicted_50k_plus_records"]].round(2)

final_tier_reconciliation = pd.DataFrame({
    "analytical_population": [len(tier_population)],
    "final_tiered_records": [final_tier_summary["listings"].sum()],
    "difference": [len(tier_population) - final_tier_summary["listings"].sum()],
    "all_predicted_50k_plus_in_tier_3": [(tier_population.loc[predicted_50k_plus, "guidance_tier_final"] == "Tier 3 — Manual appraisal").all()],
})

print("Final guidance tier sizing")
display(final_tier_summary)
print("Final tier reconciliation")
display(final_tier_reconciliation)

In [ ]:
# re-anchor the joint schedule to a fully-supported cell (age 1 / 10k miles = 100)
reference_row = joint_age_mileage_schedule[
    (joint_age_mileage_schedule["approximate_vehicle_age"] == 1)
    & (joint_age_mileage_schedule["mileage_reference_miles"] == 10000)]
assert len(reference_row) == 1
reference_index = float(reference_row["joint_adjusted_price_index"].iloc[0])
reference_support = int(reference_row["training_fingerprint_support"].iloc[0])
assert reference_support >= 100

joint_schedule_reanchored = joint_age_mileage_schedule.copy()
joint_schedule_reanchored["supported_reference_price_index"] = (
    100 * joint_schedule_reanchored["joint_adjusted_price_index"] / reference_index).round(1)
joint_schedule_reanchored["presentation_index"] = np.where(
    joint_schedule_reanchored["training_fingerprint_support"] >= 25,
    joint_schedule_reanchored["supported_reference_price_index"], np.nan)

supported_joint_matrix = joint_schedule_reanchored.pivot(
    index="approximate_vehicle_age", columns="mileage_reference_miles", values="presentation_index")
supported_joint_matrix.columns = [f"{int(m):,} miles" for m in supported_joint_matrix.columns]

headline_combinations = [(1, 10000), (3, 30000), (5, 60000), (7, 60000), (10, 100000)]
headline_joint_schedule = (joint_schedule_reanchored[
    joint_schedule_reanchored.apply(lambda r: (r["approximate_vehicle_age"], r["mileage_reference_miles"]) in headline_combinations, axis=1)]
    [["approximate_vehicle_age", "model_year", "mileage_reference_miles", "supported_reference_price_index",
      "training_fingerprint_support", "support_status"]]
    .sort_values(["approximate_vehicle_age", "mileage_reference_miles"]))

print(f"Reference: approximate age 1 and 10,000 miles = 100 (support: {reference_support})")
print("Supported joint price-index matrix")
display(supported_joint_matrix)
print("Headline supported examples")
display(headline_joint_schedule)